# Compliance Q&A — a Retrieval-Augmented Generation project

Welcome! In this project you will build a **RAG application** that lets employees ask questions
about company policy — *"How many days of annual leave do I get?"*, *"Can I accept a CHF 120 gift
from a vendor?"* — and get answers grounded in a real document set.

You will implement the **core of the pipeline yourself**. The surrounding scaffolding (data
models, the vector store, the AI clients, a web app) is provided, so you can focus on the ideas
that make RAG work.

## How this notebook works

The project ships as a normal Python repo. Rather than editing those files directly, you will
write each function **here, in the notebook**, and *monkey-patch* it onto the real project code:

```python
import chunking
def sliding_window(...): ...
chunking.sliding_window = sliding_window   # <- your version now runs everywhere
```

Because the rest of the code looks your function up by name at call-time, your implementation
immediately flows through the whole pipeline and the provided tests. At the end of each part an
**export cell** writes your functions into the project's `solutions/` folder, which is what the
full app runs on.

Those eight functions are **empty in the repo** — their bodies raise `NotImplementedError`. There
is no reference version hiding behind them, so the project's own test suite is red and the web app
cannot answer a question until you write them. That is deliberate: the pipeline runs on *your*
code or it doesn't run at all.

## Roadmap

- **Part 1 — Ingestion:** turn documents into searchable, embedded chunks — chunking, embeddings,
  and the metadata that lets you narrow a search.
- **Part 2 — RAG core:** answer a question — keyword search, embedding search, fusing the two,
  **measuring** whether any of it works, and finally calling the LLM with the retrieved context.
- **Part 3 — The app:** drop your code into the running web application.

Each step is: a short explanation → a function for **you** to complete → a **test cell** that
tells you whether it works. Eight functions in total; everything else is provided.

You never start from a blank page. Each exercise hands you the function with its structure intact
and the interesting parts punched out as `TODO` blanks, each with a one-line hint next to it:

```python
step = TODO                    # ✏️ how far the window slides between chunks
```

Fill in every `TODO`, run the cell, then run the small patch cell under it (that's what binds your
version onto the real project code) and the test cell below that.

One habit this notebook tries hard to build: **§2.5 measures every design decision you made**
rather than asserting it. Retrieval quality is an empirical question, and the difference between an
engineer and an enthusiast is having the number.

### The whole system at a glance

In [6]:
#@title 🗺️ The whole system at a glance { display-mode: "form" }
from IPython.display import HTML, display
display(HTML('<div style="text-align:center;margin:10px 0"><svg viewBox="0 0 820 300" width="100%" style="max-width:820px;font-family:ui-sans-serif,system-ui,sans-serif"><defs><marker id="ah" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L8,3 L0,6 Z" fill="#647084"/></marker></defs><text x="20" y="22" text-anchor="start" font-size="12" font-weight="700" fill="#334155">1 · WRITE — ingestion</text><rect x="20" y="46" width="150" height="54" rx="10" fill="#ffffff" stroke="#334155" stroke-width="1.6"/><text x="95.0" y="69" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Documents</text><text x="95.0" y="86" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">policies</text><rect x="220" y="46" width="150" height="54" rx="10" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/><text x="295.0" y="69" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Chunk</text><text x="295.0" y="86" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">split into passages</text><rect x="410" y="46" width="150" height="54" rx="10" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/><text x="485.0" y="69" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Embed</text><text x="485.0" y="86" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">text → vectors</text><rect x="600" y="46" width="150" height="54" rx="10" fill="#ffffff" stroke="#b45309" stroke-width="1.6"/><text x="675.0" y="69" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Vector store</text><text x="675.0" y="86" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">Qdrant</text><path d="M170,73 L220,73" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M370,73 L410,73" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M560,73 L600,73" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><text x="20" y="216" text-anchor="start" font-size="12" font-weight="700" fill="#334155">2 · READ — query</text><rect x="20" y="240" width="150" height="54" rx="10" fill="#ffffff" stroke="#334155" stroke-width="1.6"/><text x="95.0" y="272.0" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Question</text><rect x="220" y="240" width="150" height="54" rx="10" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/><text x="295.0" y="263" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Hybrid search</text><text x="295.0" y="280" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">keyword + embedding</text><rect x="470" y="240" width="120" height="54" rx="10" fill="#ffffff" stroke="#334155" stroke-width="1.6"/><text x="530.0" y="272.0" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">LLM</text><rect x="640" y="240" width="120" height="54" rx="10" fill="#ffffff" stroke="#b45309" stroke-width="1.6"/><text x="700.0" y="263" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Answer</text><text x="700.0" y="280" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">+ citations</text><path d="M170,267 L220,267" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M370,267 L470,267" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M590,267 L640,267" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M675,100 L675,150 L295,150 L295,240" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)" stroke-dasharray="5 4"/><text x="485" y="143" text-anchor="middle" font-size="10" font-weight="600" fill="#647084">stored chunks loaded into the retriever</text></svg></div>'))

The **write** path (top) is built in **Part 1**; the **read** path (bottom) in **Part 2**. The two
meet at the **vector store**: ingestion writes embedded chunks into it, and every query loads those
chunks back to search them.

## 0.1 — Setup

Run the cell below first. In **Colab** it clones the course repo (public — no token needed),
installs the dependencies, and puts the project's code on the import path. Running **locally
inside the repo** it just makes the project importable, whether you launched Jupyter from
`project/` or from `project/notebook/`.

The project lives in the **`project/`** folder of the course repo, so that is the working
directory everything below assumes — `data/`, `solutions/` and `tests/` are relative to it.

The only credential this notebook ever reads is an *optional* OpenRouter key, and it comes from
**Colab Secrets** (next section) — never pasted into a cell.

In [7]:
#@title 🔧 Setup — clone the course repo and install dependencies { display-mode: "form" }
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

# The course repo. It is public, so no token and no authentication are needed.
REPO_URL = "https://github.com/eth-fdd-fs26/FDD-WE5-public.git"
REPO_BRANCH = "main"
REPO_DIR = "FDD-WE5-public"
# Everything for this project lives in one folder of that repo.
PROJECT_SUBDIR = "project"


def colab_secret(name):
    """Read a Colab secret by name; return None if missing or access not granted."""
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        # --branch pins the branch; --single-branch --depth 1 keeps the clone small.
        # We capture output rather than check=True so a failure prints a readable
        # message instead of a raw CalledProcessError.
        proc = subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", "--depth", "1",
             REPO_URL, REPO_DIR],
            capture_output=True, text=True,
        )
        if proc.returncode != 0:
            raise RuntimeError(
                "git clone failed (exit %d):\n%s\n\nCheck that REPO_URL (%r) and "
                "REPO_BRANCH (%r) are right, then re-run this cell."
                % (proc.returncode, (proc.stderr or proc.stdout or "").strip(),
                   REPO_URL, REPO_BRANCH)
            )
    %pip install -q qdrant-client rank-bm25 numpy openai pypdf rich
    project = os.path.abspath(os.path.join(REPO_DIR, PROJECT_SUBDIR))
    sys.path.insert(0, project)
    os.chdir(project)
else:
    # Local: walk up until we find the project folder (the one holding chunking.py),
    # make it importable, and work from there — so it doesn't matter whether you
    # launched Jupyter from project/ or from project/notebook/.
    root = os.path.abspath(".")
    while root != os.path.dirname(root) and not os.path.exists(os.path.join(root, "chunking.py")):
        root = os.path.dirname(root)
    if not os.path.exists(os.path.join(root, "chunking.py")):
        raise RuntimeError(
            "Could not find the project folder (the one containing chunking.py). "
            "Run this notebook from inside the repo's project/ folder."
        )
    sys.path.insert(0, root)
    os.chdir(root)

print("Setup complete. Running in Colab:", IN_COLAB)
print("Working directory:", os.getcwd())

Setup complete. Running in Colab: False
Working directory: c:\Users\stefa\Documents\0000_ETH\275-0005-00L AI Workshop - From Data to Solutions\Project 5 - Compliance Q&A\GitHub\project


## 0.2 — An (optional) OpenRouter key

Everything in this notebook runs **fully offline** using small mock AI clients, so you do not need
an API key to do the project or pass the tests.

If you *do* want to see the real system answer a question at the end, add your
[OpenRouter](https://openrouter.ai) key as the Colab secret **`OPENROUTER_API_KEY`** (🔑 Secrets
panel, "Notebook access" ON). The cell below loads it from your secrets — locally, you can instead
`export OPENROUTER_API_KEY=...` before launching Jupyter. The key is read straight into the
environment and never written into the notebook.

In [8]:
from pathlib import Path
env_file = Path(".env")  # cwd is already project/ after the 0.1 setup cell chdir's into it
if env_file.exists():
    for line in env_file.read_text().splitlines():
        if line.strip() and not line.startswith("#"):
            k, _, v = line.partition("=")
            os.environ.setdefault(k.strip(), v.strip().strip("'\""))

In [9]:
# Pull the OpenRouter key from Colab Secrets (no-op locally / if the secret is absent).
_k = colab_secret("OPENROUTER_API_KEY") if IN_COLAB else None
if _k:
    os.environ["OPENROUTER_API_KEY"] = _k

HAS_KEY = bool(os.environ.get("OPENROUTER_API_KEY"))
print("Live API key available:", HAS_KEY)
print("(Everything below works offline with mocks regardless.)")

Live API key available: True
(Everything below works offline with mocks regardless.)


## 0.3 — Imports

We import the provided scaffolding once. Notice we reuse the project's own test harness
(`TestSuite`) and offline doubles (`MockEmbedder`, `MockLLM`) — the exact same tools the course
maintainers use.

In [11]:
import inspect
import pathlib

import numpy as np
from rank_bm25 import BM25Okapi

# Provided scaffolding from the repo:
import chunking
import loaders
from ingestion_core import IngestionCore
from db_manager import DBManager
from rag_core import NO_CONTEXT_ANSWER, RAGCore
from retrieval_core import RetrievalCore, SearchType, _tokenize
from models import Chunk, Document

import evaluation as ev

# Offline test doubles + the friendly test runner (also from the repo):
# MockEmbedder returns one fixed vector (perfect for deterministic unit tests);
# HashingEmbedder gives every text its own vector, so demos and measurements
# offline are meaningful rather than a pile of ties.
from tests.mocks import HashingEmbedder, MockEmbedder, MockLLM
from tests.harness import TestSuite

print("Imports OK — ready to build the pipeline.")

Imports OK — ready to build the pipeline.


### The `TODO` blank

Every exercise arrives as **working code with holes in it**. The scaffolding — the loop, the guard
clauses, the order of operations — is already written; the parts that carry the actual idea are
replaced by the marker `TODO`:

```python
step = TODO                    # ✏️ how far the window slides between chunks
```

Your job is to replace each `TODO` with the missing expression. `TODO` is a live object, not a
comment: touching one in any way (calling it, comparing it, doing arithmetic with it, iterating it)
raises immediately, so a blank you forgot can never quietly pass as a wrong answer. Run the cell
below once to define it.

In [12]:
#@title 🔧 The TODO blank marker — run, don't read { display-mode: "form" }
class _Blank:
    """A hole in the exercise code. Any use raises, so an unfilled blank fails loudly."""

    _MSG = ("✏️ Unfilled blank. Replace every `TODO` in the exercise cell above with real "
            "code, then re-run that cell (and the patch cell after it).")

    def _blank(self, *args, **kwargs):
        raise NotImplementedError(self._MSG)

    def __getattr__(self, name):
        if name.startswith("_"):        # let Python/IPython probe for dunders quietly
            raise AttributeError(name)
        self._blank()

    def __repr__(self):                 # safe: displaying a TODO should explain, not explode
        return "TODO  # ✏️ still to fill in"

    __call__ = __iter__ = __len__ = __bool__ = __hash__ = _blank
    __index__ = __int__ = __float__ = __str__ = __format__ = _blank
    __eq__ = __ne__ = __lt__ = __le__ = __gt__ = __ge__ = _blank
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = _blank
    __truediv__ = __rtruediv__ = __neg__ = __matmul__ = __rmatmul__ = _blank
    __contains__ = __getitem__ = __setitem__ = __array__ = _blank

TODO = _Blank()


def has_blanks(fn):
    """True if `fn`'s source still mentions TODO — used to keep stubs out of the export."""
    return "TODO" in inspect.getsource(fn)


print("TODO marker ready. Blanks look like:", repr(TODO))

TODO marker ready. Blanks look like: TODO  # ✏️ still to fill in


## 0.4 — Pretty output helpers

To make results easy to read, we render them as small HTML cards instead of plain text. Just run
this cell — you'll call these helpers (`show_documents`, `show_chunks`, `show_results`,
`show_answer`) throughout the notebook.

In [13]:
#@title 🔧 Display helpers — run, don't read { display-mode: "form" }
from IPython.display import HTML, display

# A small, consistent visual language for outputs in this notebook.
_INK, _MUTE, _TEAL, _AMBER, _LINE, _BG = "#1f2933", "#647084", "#0f766e", "#b45309", "#e3e8ef", "#f8fafc"
_FONT = "ui-sans-serif,system-ui,sans-serif"
_MONO = "ui-monospace,SFMono-Regular,Menlo,monospace"

def _card(inner, accent=_TEAL):
    return (f'<div style="border:1px solid {_LINE};border-left:4px solid {accent};border-radius:10px;'
            f'padding:14px 16px;margin:8px 0;background:#fff;font-family:{_FONT}">{inner}</div>')

def _clip(text, n):
    return (text[:n] + "…") if len(text) > n else text

def show_documents(docs):
    head = (f'<tr style="color:{_MUTE};font-size:11px;text-transform:uppercase;letter-spacing:.05em">'
            f'<th style="text-align:left;padding:6px 10px">Document</th>'
            f'<th style="text-align:left;padding:6px 10px">Source</th>'
            f'<th style="text-align:right;padding:6px 10px">Words</th></tr>')
    rows = "".join(
        f'<tr><td style="padding:6px 10px;border-top:1px solid {_LINE};font-weight:600;'
        f'color:{_INK};text-align:left">{d.metadata["document_title"]}</td>'
        f'<td style="padding:6px 10px;border-top:1px solid {_LINE};color:{_MUTE};'
        f'font-family:{_MONO};font-size:12px;text-align:left">{d.metadata["source"]}</td>'
        f'<td style="padding:6px 10px;border-top:1px solid {_LINE};text-align:right;color:{_INK}">{len(d.text.split())}</td></tr>'
        for d in docs)
    title = f'<div style="font-weight:700;color:{_INK};margin-bottom:6px">📚 {len(docs)} policy documents</div>'
    display(HTML(_card(title + f'<table style="border-collapse:collapse;width:100%">{head}{rows}</table>')))

def show_chunks(chunks, title="Chunks", limit=4):
    items = "".join(
        f'<div style="background:{_BG};border:1px solid {_LINE};border-radius:8px;padding:8px 10px;margin:6px 0">'
        f'<span style="font-family:{_MONO};font-size:11px;color:{_TEAL};font-weight:700">#{c.index}</span> '
        f'<span style="color:{_INK};font-size:13px">{_clip(c.text, 150)}</span></div>'
        for c in chunks[:limit])
    more = f'<div style="color:{_MUTE};font-size:12px">… and {len(chunks)-limit} more</div>' if len(chunks) > limit else ""
    display(HTML(_card(f'<div style="font-weight:700;color:{_INK};margin-bottom:4px">🧩 {title} · {len(chunks)} total</div>{items}{more}')))

def show_chunking(whole, windows):
    def col(name, chunks, accent):
        items = "".join(
            f'<div style="background:{_BG};border:1px solid {_LINE};border-radius:8px;padding:8px;margin:6px 0;font-size:12px;color:{_INK}">{_clip(ch, 120)}</div>'
            for ch in chunks[:4])
        more = f'<div style="color:{_MUTE};font-size:12px">… {len(chunks)-4} more</div>' if len(chunks) > 4 else ""
        return f'<div style="flex:1"><div style="font-weight:700;color:{accent};margin-bottom:4px">{name} · {len(chunks)} chunk(s)</div>{items}{more}</div>'
    display(HTML(_card(f'<div style="display:flex;gap:16px">{col("whole_document", whole, _AMBER)}{col("sliding_window", windows, _TEAL)}</div>')))

def show_results(results, title="Results"):
    if not results:
        display(HTML(_card("<em>No results.</em>", _AMBER)))
        return
    mx = max(s for _, s in results) or 1.0
    rows = []
    for rank, (c, s) in enumerate(results, 1):
        pct = max(4.0, 100 * s / mx)
        rows.append(
            f'<div style="display:flex;gap:10px;align-items:flex-start;padding:8px 0;border-top:1px solid {_LINE}">'
            f'<div style="color:{_MUTE};font-weight:700;width:20px">{rank}</div><div style="flex:1">'
            f'<div style="font-size:11px;color:{_MUTE};font-family:{_MONO}">{c.metadata.get("source", "?")}</div>'
            f'<div style="color:{_INK};font-size:13px;margin:2px 0 4px">{_clip(c.text, 150)}</div>'
            f'<div style="background:{_BG};border-radius:6px;height:6px;overflow:hidden">'
            f'<div style="width:{pct:.0f}%;height:6px;background:{_TEAL}"></div></div></div>'
            f'<div style="font-family:{_MONO};color:{_TEAL};font-weight:700">{s:.3f}</div></div>')
    display(HTML(_card(f'<div style="font-weight:700;color:{_INK};margin-bottom:2px">🔎 {title}</div>' + "".join(rows))))

def show_answer(answer, sources):
    src = "".join(
        f'<li style="margin:5px 0;color:{_MUTE};font-size:13px"><code style="color:{_TEAL}">{c.metadata.get("source", "?")}</code> '
        f'<span style="color:{_TEAL};font-weight:700">{s:.3f}</span> — {_clip(c.text, 90)}</li>'
        for c, s in sources)
    body = (f'<div style="font-size:11px;color:{_MUTE};text-transform:uppercase;letter-spacing:.05em">Answer</div>'
            f'<div style="color:{_INK};font-size:15px;line-height:1.55;margin:6px 0 10px">{answer}</div>'
            f'<details><summary style="cursor:pointer;color:{_TEAL};font-weight:700">{len(sources)} sources</summary>'
            f'<ul style="margin:8px 0 0;padding-left:18px">{src}</ul></details>')
    display(HTML(_card(body, _AMBER)))

print("Display helpers ready: show_documents, show_chunks, show_chunking, show_results, show_answer")

Display helpers ready: show_documents, show_chunks, show_chunking, show_results, show_answer


## 0.5 — Evaluation helpers

Part 2 stops guessing and starts **measuring**. These render the results: a scoreboard comparing
configurations, a per-question grid showing exactly which questions failed, and side-by-side
retrieval panels. Run and move on — you'll call them, not read them.

In [14]:
#@title 🔧 Evaluation helpers — run, don't read { display-mode: "form" }
_GREEN, _RED, _VIOLET = "#15803d", "#b91c1c", "#7c3aed"

def explain(title, body, tone="info", icon="💡"):
    """A callout card. `body` is raw HTML."""
    edge = {"info": _TEAL, "warn": _AMBER, "bad": _RED, "good": _GREEN}[tone]
    display(HTML(
        f'<div style="border:1px solid {_LINE};border-left:5px solid {edge};border-radius:12px;'
        f'padding:13px 16px;margin:8px 0;background:linear-gradient(135deg,#fbfdfd,#f7fafc);font-family:{_FONT}">'
        f'<div style="font-weight:750;color:{_INK};font-size:14.5px;margin-bottom:5px">{icon} {title}</div>'
        f'<div style="color:#3d4753;font-size:13px;line-height:1.6">{body}</div></div>'))

def _bar(pct, colour, width=110):
    pct = max(0.0, min(100.0, pct))
    return (f'<span style="display:inline-block;width:{width}px;height:8px;background:{_BG};'
            f'border:1px solid {_LINE};border-radius:5px;overflow:hidden;vertical-align:middle">'
            f'<span style="display:block;height:100%;width:{pct:.0f}%;background:{colour}"></span></span>')

def show_compare(panels, note=""):
    """Side-by-side ranked result lists. `panels` = [(title, results, gold_source_or_None), ...]"""
    cols = []
    for title, results, gold in panels:
        rows = ""
        for rank, (c, s) in enumerate(results, 1):
            src = c.metadata.get("source", "?")
            is_gold = gold is not None and src == gold
            edge = _GREEN if is_gold else _LINE
            tag = (f'<span style="background:#dcfce7;color:{_GREEN};font-size:9px;font-weight:800;'
                   f'border-radius:4px;padding:1px 5px;margin-left:5px">GOLD</span>') if is_gold else ""
            rows += (f'<div style="border:1px solid {_LINE};border-left:3px solid {edge};border-radius:8px;'
                     f'padding:6px 9px;margin:5px 0;background:#fff">'
                     f'<div style="font-size:10.5px;color:{_MUTE};font-family:{_MONO}">'
                     f'<b style="color:{_TEAL}">#{rank}</b> {src}{tag}'
                     f'<span style="float:right;color:{_TEAL};font-weight:700">{s:.3f}</span></div>'
                     f'<div style="color:{_INK};font-size:12px;margin-top:3px;line-height:1.45">{_clip(c.text, 130)}</div></div>')
        if not results:
            rows = f'<div style="color:{_AMBER};font-size:12.5px;padding:8px 0"><em>No results.</em></div>'
        cols.append(f'<div style="flex:1;min-width:0"><div style="font-weight:700;color:{_INK};'
                    f'font-size:12.5px;margin-bottom:4px">{title}</div>{rows}</div>')
    tail = f'<div style="color:{_MUTE};font-size:12px;margin-top:8px">{note}</div>' if note else ""
    display(HTML(_card(f'<div style="display:flex;gap:14px;align-items:flex-start">{"".join(cols)}</div>{tail}')))

def show_eval_grid(rows, title="Per-question results", subtitle=""):
    """`rows` = output of evaluation.per_query_hits(): [(EvalQuery, hit, top_source), ...]"""
    items = ""
    for q, hit, top in rows:
        colour, icon = (_GREEN, "✓") if hit else (_RED, "✗")
        got = "" if hit else (f'<div style="font-size:11px;color:{_MUTE};margin-top:2px">'
                              f'top hit was <code style="color:{_AMBER}">{top or "nothing"}</code></div>')
        items += (f'<div style="display:flex;gap:9px;align-items:flex-start;border:1px solid {_LINE};'
                  f'border-left:3px solid {colour};border-radius:8px;padding:6px 10px;margin:5px 0;background:#fff">'
                  f'<span style="color:{colour};font-weight:800;font-size:14px;width:12px">{icon}</span>'
                  f'<div style="flex:1;min-width:0">'
                  f'<div style="color:{_INK};font-size:12.5px">{q.question}</div>'
                  f'<div style="font-size:10.5px;color:{_MUTE};font-family:{_MONO};margin-top:2px">'
                  f'{q.kind} · expects {q.gold_source}</div>{got}</div></div>')
    n = sum(1 for _, hit, _ in rows if hit)
    ring = _GREEN if n == len(rows) else (_AMBER if n >= len(rows) * 0.6 else _RED)
    head = (f'<div style="display:flex;align-items:baseline;gap:10px">'
            f'<div style="font-weight:700;color:{_INK}">🎯 {title}</div>'
            f'<div style="margin-left:auto;font-size:20px;font-weight:800;color:{ring}">{n}/{len(rows)}</div></div>'
            f'<div style="color:{_MUTE};font-size:12px;margin:1px 0 6px">{subtitle}</div>')
    display(HTML(_card(head + items, ring)))

def show_scoreboard(rows, title="Scoreboard", subtitle="", cost_key="words"):
    """`rows` = [{"label": str, "doc": float, "answer": float, "words": float}, ...]

    Highlights the best value in each column. Cost is best when *lowest*.
    """
    best_doc = max(r["doc"] for r in rows)
    best_ans = max(r["answer"] for r in rows)
    best_cost = min(r[cost_key] for r in rows)
    head = (f'<tr style="color:{_MUTE};font-size:10.5px;text-transform:uppercase;letter-spacing:.05em">'
            f'<th style="text-align:left;padding:5px 8px">Configuration</th>'
            f'<th style="text-align:left;padding:5px 8px">Document recall</th>'
            f'<th style="text-align:left;padding:5px 8px">Answer recall</th>'
            f'<th style="text-align:right;padding:5px 8px">Context words</th></tr>')
    body = ""
    for r in rows:
        def cell(value, best, colour):
            weight = "800" if value == best else "500"
            mark = " ★" if value == best else ""
            return (f'<td style="padding:6px 8px;border-top:1px solid {_LINE};white-space:nowrap;'
                    f'text-align:left">'
                    f'{_bar(value * 100, colour)} <span style="font-family:{_MONO};font-size:11.5px;'
                    f'color:{_INK};font-weight:{weight}">{value:.2f}{mark}</span></td>')
        cost = r[cost_key]
        cheapest = cost == best_cost
        body += (f'<tr><td style="padding:6px 8px;border-top:1px solid {_LINE};font-weight:650;'
                 f'color:{_INK};font-size:12.5px;text-align:left">{r["label"]}</td>'
                 + cell(r["doc"], best_doc, _TEAL) + cell(r["answer"], best_ans, _VIOLET)
                 + f'<td style="padding:6px 8px;border-top:1px solid {_LINE};text-align:right;'
                   f'font-family:{_MONO};font-size:11.5px;color:{_GREEN if cheapest else _INK};'
                   f'font-weight:{"800" if cheapest else "500"}">{cost:,.0f}{" ★" if cheapest else ""}</td></tr>')
    head_html = (f'<div style="font-weight:700;color:{_INK}">🏆 {title}</div>'
                 f'<div style="color:{_MUTE};font-size:12px;margin:1px 0 6px">{subtitle}</div>')
    display(HTML(_card(head_html + f'<table style="border-collapse:collapse;width:100%">{head}{body}</table>')))

def show_kinds(scores, title="Recall by question kind", subtitle=""):
    """`scores` = {"lexical": float, "semantic": float, "overall": float}"""
    order = [("lexical", "Lexical questions", _TEAL), ("semantic", "Semantic questions", _VIOLET),
             ("overall", "All questions", _INK)]
    rows = "".join(
        f'<div style="display:flex;align-items:center;gap:10px;margin:6px 0">'
        f'<div style="width:150px;color:{_INK};font-size:12.5px">{label}</div>'
        f'{_bar(scores[key] * 100, colour, 200)}'
        f'<div style="font-family:{_MONO};font-weight:700;color:{colour}">{scores[key]:.2f}</div></div>'
        for key, label, colour in order if key in scores)
    head = (f'<div style="font-weight:700;color:{_INK}">📊 {title}</div>'
            f'<div style="color:{_MUTE};font-size:12px;margin:1px 0 6px">{subtitle}</div>')
    display(HTML(_card(head + rows)))

def show_sweep(rows, title, x_label, subtitle=""):
    """A tiny column chart. `rows` = [(label, value, cost_or_None), ...] with value in 0..1."""
    top = max(v for _, v, _ in rows) or 1.0
    cols = ""
    for label, value, cost in rows:
        height = max(6, 96 * value / top)
        best = value == top
        colour = _TEAL if best else "#94a3b8"
        cost_html = (f'<div style="font-size:9.5px;color:{_MUTE};font-family:{_MONO}">{cost:,.0f}w</div>'
                     if cost is not None else "")
        cols += (f'<div style="flex:1;text-align:center;min-width:0">'
                 f'<div style="font-family:{_MONO};font-size:11px;color:{colour};font-weight:700">{value:.2f}</div>'
                 f'<div style="height:100px;display:flex;align-items:flex-end;justify-content:center">'
                 f'<div style="width:70%;height:{height:.0f}px;background:{colour};border-radius:4px 4px 0 0"></div></div>'
                 f'<div style="border-top:1px solid {_LINE};padding-top:4px;font-size:11px;color:{_INK}">{label}</div>'
                 f'{cost_html}</div>')
    head = (f'<div style="font-weight:700;color:{_INK}">📈 {title}</div>'
            f'<div style="color:{_MUTE};font-size:12px;margin:1px 0 8px">{subtitle}</div>')
    foot = f'<div style="text-align:center;color:{_MUTE};font-size:11px;margin-top:6px">{x_label}</div>'
    display(HTML(_card(head + f'<div style="display:flex;gap:8px;align-items:flex-end">{cols}</div>' + foot)))

print("Evaluation helpers ready: explain, show_compare, show_eval_grid, show_scoreboard, show_kinds, show_sweep")

Evaluation helpers ready: explain, show_compare, show_eval_grid, show_scoreboard, show_kinds, show_sweep


---
# Part 1 — Document ingestion pipeline

Ingestion is the **write side** of RAG. It turns raw documents into searchable chunks:

```
load  ->  CHUNK  ->  EMBED  ->  store
```

1. **load** — read a file into a `Document` (provided), attaching metadata.
2. **chunk** — split the text into focused passages (**you implement this**).
3. **embed** — turn each chunk into a vector (**you wire this up**).
4. **store** — persist the embedded chunks into the vector database (provided).

In [15]:
#@title 📥 The ingestion pipeline { display-mode: "form" }
from IPython.display import HTML, display
display(HTML('<div style="text-align:center;margin:10px 0"><svg viewBox="0 0 770 140" width="100%" style="max-width:770px;font-family:ui-sans-serif,system-ui,sans-serif"><defs><marker id="ah" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L8,3 L0,6 Z" fill="#647084"/></marker></defs><rect x="20" y="46" width="160" height="54" rx="10" fill="#ffffff" stroke="#334155" stroke-width="1.6"/><text x="100.0" y="69" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">load</text><text x="100.0" y="86" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">file → Document</text><rect x="210" y="46" width="160" height="54" rx="10" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/><text x="290.0" y="69" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">CHUNK</text><text x="290.0" y="86" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">Document → texts</text><rect x="400" y="46" width="160" height="54" rx="10" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/><text x="480.0" y="69" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">EMBED</text><text x="480.0" y="86" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">texts → vectors</text><rect x="590" y="46" width="160" height="54" rx="10" fill="#ffffff" stroke="#b45309" stroke-width="1.6"/><text x="670.0" y="69" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">store</text><text x="670.0" y="86" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">→ vector DB</text><path d="M180,73 L210,73" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M370,73 L400,73" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M560,73 L590,73" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><text x="100" y="118" text-anchor="middle" font-size="10" font-weight="600" fill="#647084">loaders.py</text><text x="290" y="118" text-anchor="middle" font-size="10" font-weight="700" fill="#0f766e">chunking.py  ✏️ you</text><text x="480" y="118" text-anchor="middle" font-size="10" font-weight="700" fill="#0f766e">ingest_document  ✏️ you</text><text x="670" y="118" text-anchor="middle" font-size="10" font-weight="600" fill="#647084">db_manager.py</text></svg></div>'))

The ✏️ marked steps are the ones **you** implement. Let's look at the documents we'll work with.

In [16]:
#@title 📚 The document set at a glance { display-mode: "form" }
docs = loaders.load_directory("data")
show_documents(docs)

Document,Source,Words
Artificial Intelligence Usage Policy,ai_usage_policy.md,1015
Code of Conduct,code_of_conduct.md,1241
Data Retention Regulation,data_retention_regulation.md,175
Expense Reimbursement Policy,expense_reimbursement_policy.md,952
Information Security Policy,information_security_policy.md,1180
Leave and Absence Policy,leave_and_absence_policy.md,1118
Performance Review Policy,performance_review_policy.md,1235
Procurement Policy,procurement_policy.md,1084
Remote Work Policy,remote_work_policy.md,170


## 1.1 — Chunking

An embedding model turns a piece of text into **one** vector. Embed a whole 800-word policy as a
single vector and it becomes a blurry average of every idea in it — a question about one clause has
to compete with the noise of the entire document.

The fix is **chunking**: split the document into smaller passages so each idea gets its own vector.
We use a **sliding window** over words, with a little **overlap** so a sentence straddling a
boundary isn't lost.

In [17]:
#@title 🪟 How a sliding window overlaps { display-mode: "form" }
from IPython.display import HTML, display
display(HTML('<div style="text-align:center;margin:10px 0"><svg viewBox="0 0 600 200" width="100%" style="max-width:600px;font-family:ui-sans-serif,system-ui,sans-serif"><defs><marker id="ah" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L8,3 L0,6 Z" fill="#647084"/></marker></defs><text x="20" y="28" text-anchor="start" font-size="11" font-weight="600" fill="#334155">step = chunk_size − overlap = 3   (chunk_size=4, overlap=1)</text><rect x="20" y="58" width="54" height="32" rx="6" fill="#f8fafc" stroke="#e3e8ef"/><text x="47" y="79" text-anchor="middle" font-size="12" font-weight="600" fill="#1f2933">w0</text><rect x="82" y="58" width="54" height="32" rx="6" fill="#f8fafc" stroke="#e3e8ef"/><text x="109" y="79" text-anchor="middle" font-size="12" font-weight="600" fill="#1f2933">w1</text><rect x="144" y="58" width="54" height="32" rx="6" fill="#f8fafc" stroke="#e3e8ef"/><text x="171" y="79" text-anchor="middle" font-size="12" font-weight="600" fill="#1f2933">w2</text><rect x="206" y="58" width="54" height="32" rx="6" fill="#f8fafc" stroke="#e3e8ef"/><text x="233" y="79" text-anchor="middle" font-size="12" font-weight="600" fill="#1f2933">w3</text><rect x="268" y="58" width="54" height="32" rx="6" fill="#f8fafc" stroke="#e3e8ef"/><text x="295" y="79" text-anchor="middle" font-size="12" font-weight="600" fill="#1f2933">w4</text><rect x="330" y="58" width="54" height="32" rx="6" fill="#f8fafc" stroke="#e3e8ef"/><text x="357" y="79" text-anchor="middle" font-size="12" font-weight="600" fill="#1f2933">w5</text><rect x="392" y="58" width="54" height="32" rx="6" fill="#f8fafc" stroke="#e3e8ef"/><text x="419" y="79" text-anchor="middle" font-size="12" font-weight="600" fill="#1f2933">w6</text><rect x="454" y="58" width="54" height="32" rx="6" fill="#f8fafc" stroke="#e3e8ef"/><text x="481" y="79" text-anchor="middle" font-size="12" font-weight="600" fill="#1f2933">w7</text><rect x="516" y="58" width="54" height="32" rx="6" fill="#f8fafc" stroke="#e3e8ef"/><text x="543" y="79" text-anchor="middle" font-size="12" font-weight="600" fill="#1f2933">w8</text><rect x="16" y="104" width="250" height="24" rx="6" fill="#0f766e" fill-opacity="0.16" stroke="#0f766e" stroke-width="1.4"/><text x="141.0" y="120" text-anchor="middle" font-size="10.5" font-weight="700" fill="#0f766e">chunk 0  ·  w0–w3</text><rect x="202" y="134" width="250" height="24" rx="6" fill="#b45309" fill-opacity="0.16" stroke="#b45309" stroke-width="1.4"/><text x="327.0" y="150" text-anchor="middle" font-size="10.5" font-weight="700" fill="#b45309">chunk 1  ·  w3–w6</text><rect x="388" y="164" width="188" height="24" rx="6" fill="#334155" fill-opacity="0.16" stroke="#334155" stroke-width="1.4"/><text x="482.0" y="180" text-anchor="middle" font-size="10.5" font-weight="700" fill="#334155">chunk 2  ·  w6–w8</text></svg></div>'))

Notice how `w3` and `w6` each sit in **two** chunks — that's the overlap, and it's what keeps a
sentence spanning a boundary from being lost.

> **Reflection:** what goes wrong if `overlap` is large relative to `chunk_size`? What if it's 0?

In [18]:
#@title ✏️ Your task · 1.1 { display-mode: "form" }
from IPython.display import HTML, display
display(HTML('<div style="border:1px solid #e3e8ef;border-left:5px solid #0f766e;border-radius:12px;\n     background:linear-gradient(135deg,#fbfefe,#f7fafc);padding:0 0 16px;margin:14px 0;\n     font-family:ui-sans-serif,system-ui,sans-serif;max-width:920px">\n  <div style="background:#0f766e;color:#fff;padding:7px 16px;border-radius:7px 7px 0 0;\n       font-size:11px;font-weight:700;letter-spacing:.09em;text-transform:uppercase">\n    ✏️ Your task &nbsp;·&nbsp; 1.1\n  </div>\n  <div style="padding:0 16px">\n    <div style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:13.5px;color:#1f2933;font-weight:600;\n         background:#fff;border:1px solid #e3e8ef;border-radius:7px;padding:8px 11px;margin-top:13px;\n         overflow-x:auto;white-space:pre">sliding_window(text, chunk_size=200, overlap=40)</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">What it must do</div>\n    <div style="font-size:13px;color:#1f2933;line-height:1.55">Slide a fixed-size window over the <b>words</b> of <code>text</code>, stepping forward by <code>chunk_size - overlap</code> each time, and return the windows as strings. Stop as soon as a window reaches the end, so the last chunk is not a duplicate of the one before it.</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Use these — already provided</div>\n    <table style="border-collapse:collapse;width:100%"><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">text.split()</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the words to window over</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">" ".join(window)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">a window back into a chunk string</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">range(0, len(words), step)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the start index of each window</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">raise ValueError(...)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">reject chunk_size &lt;= 0, overlap &lt; 0, overlap &gt;= chunk_size</td></tr></table>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Returns</div>\n    <div style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px;color:#1f2933">list[str] — the chunk texts, never empty strings</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Where it plugs in</div>\n    <div style="font-size:12px;color:#334155">Once written you bind it onto\n      <code style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;color:#0f766e">chunking.sliding_window</code>, and the whole pipeline\n      starts using your version. The cell after it checks your work.</div>\n    <div style="margin-top:10px;padding:7px 10px;background:#fffbeb;border:1px solid #fde68a;border-radius:7px;font-size:12px;color:#78350f">⚠️ Validate the arguments first. overlap &gt;= chunk_size means the window never moves forward, which is an infinite loop rather than a bad result.</div>\n  </div>\n</div>'))

text.split(),the words to window over
""" "".join(window)",a window back into a chunk string
"range(0, len(words), step)",the start index of each window
raise ValueError(...),"reject chunk_size <= 0, overlap < 0, overlap >= chunk_size"


In [19]:
def sliding_window(text, chunk_size=200, overlap=40):
    """Split `text` into overlapping windows of words."""
    # --- validation: a window that never advances is an infinite loop, not a bad result.
    if chunk_size <= 0:
        raise ValueError(f"chunk_size must be positive, got {chunk_size}")
    if overlap < 0:
        raise ValueError(f"overlap must be >= 0, got {overlap}")
    if overlap >= chunk_size:                       # ✏️ which overlap stops the window moving forward?
        raise ValueError(f"overlap ({overlap}) must be smaller than chunk_size ({chunk_size})")

    words = text.split()
    if not words:
        return []

    step = chunk_size - overlap                     # ✏️ how far the window slides between chunks
    chunks = []
    for start in range(0, len(words), step):
        window = words[start:start + chunk_size]    # ✏️ where does this window end?
        if not window:
            break
        chunks.append(" ".join(window))             # ✏️ the window as one space-joined string
        if start + chunk_size >= len(words):        # ✏️ this window already reached the last word -> stop,
            break                                   #    otherwise the tail chunk is emitted twice
    return chunks

In [20]:
chunking.sliding_window = sliding_window
print("Patched chunking.sliding_window ->", chunking.sliding_window.__name__)

Patched chunking.sliding_window -> sliding_window


In [21]:
suite = TestSuite("Part 1.1 — sliding_window")

@suite.case("sliding_window", "overlap shares words across consecutive windows")
def _():
    assert sliding_window("a b c d e f", chunk_size=4, overlap=1) == ["a b c d", "d e f"]

@suite.case("sliding_window", "text shorter than the window -> a single chunk")
def _():
    assert sliding_window("a b", chunk_size=4, overlap=1) == ["a b"]

@suite.case("sliding_window", "empty text -> no chunks")
def _():
    assert sliding_window("   ", chunk_size=4, overlap=1) == []

@suite.case("sliding_window", "overlap >= chunk_size is rejected")
def _():
    try:
        sliding_window("a b c", chunk_size=2, overlap=2)
        raise AssertionError("expected ValueError")
    except ValueError:
        pass

suite.run()

╭───────────────────────────╮
│ Part 1.1 — sliding_window │
╰───────────────────────────╯

sliding_window  4/4

✓ overlap shares words across consecutive windows

✓ text shorter than the window -> a single chunk

✓ empty text -> no chunks

✓ overlap >= chunk_size is rejected

              Summary               
                                    
  Function         Passed   Status  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  sliding_window    4/4     ✓ PASS

╭─────────────────────────────────────╮
│ All 4 checks passed — nice work! 🎉 │
╰─────────────────────────────────────╯

True

**See why chunking matters.** Compare the "no chunking" baseline against your sliding window on a
real policy — notice how many focused passages you get instead of one giant blob.

In [22]:
#@title 🧩 Chunking side by side — no chunking vs your sliding window { display-mode: "form" }
sample = next(d for d in docs if "leave" in d.metadata["source"])
whole = chunking.chunk_text(sample.text, strategy="whole_document")
windows = chunking.chunk_text(sample.text, chunk_size=80, overlap=15, strategy="sliding_window")
show_chunking(whole, windows)

## 1.2 — Embeddings & the simple vector database

An **embedding** maps text to a vector so that *similar meanings land near each other*. Once every
chunk is a vector, "search" becomes "find the nearest vectors to the query vector".

You'll wire up the full ingestion step on `IngestionCore`: chunk the document, wrap each passage in
a `Chunk`, embed them all in one batched call, and store them. The `embedder`, `db`, and chunking
config are already on `self`.

- `chunking.chunk_text(text, chunk_size=..., overlap=..., strategy=...) -> list[str]`
- `Chunk(document_id=..., index=..., text=..., metadata=...)`
- `self.embedder.embed_batch(list_of_texts) -> list[list[float]]` (order preserved)
- `self.db.ensure_collection(name, vector_size=...)` then `self.db.insert_document(name, document, chunks)`

In [23]:
#@title ✏️ Your task · 1.2 { display-mode: "form" }
from IPython.display import HTML, display
display(HTML('<div style="border:1px solid #e3e8ef;border-left:5px solid #0f766e;border-radius:12px;\n     background:linear-gradient(135deg,#fbfefe,#f7fafc);padding:0 0 16px;margin:14px 0;\n     font-family:ui-sans-serif,system-ui,sans-serif;max-width:920px">\n  <div style="background:#0f766e;color:#fff;padding:7px 16px;border-radius:7px 7px 0 0;\n       font-size:11px;font-weight:700;letter-spacing:.09em;text-transform:uppercase">\n    ✏️ Your task &nbsp;·&nbsp; 1.2\n  </div>\n  <div style="padding:0 16px">\n    <div style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:13.5px;color:#1f2933;font-weight:600;\n         background:#fff;border:1px solid #e3e8ef;border-radius:7px;padding:8px 11px;margin-top:13px;\n         overflow-x:auto;white-space:pre">ingest_document(self, document)</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">What it must do</div>\n    <div style="font-size:13px;color:#1f2933;line-height:1.55">Run the whole write side for one document: chunk its text, wrap each passage in a <code>Chunk</code> that carries the document\'s metadata, embed them all in <b>one</b> batched call, and persist them. Return the chunks you stored.</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Use these — already provided</div>\n    <table style="border-collapse:collapse;width:100%"><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">chunking.chunk_text(text, chunk_size=, overlap=, strategy=)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the passages to store</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">Chunk(document_id=, index=, text=, metadata=)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">one chunk object per passage</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">self.embedder.embed_batch(texts)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">one vector per text, order preserved</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">self.db.ensure_collection(name, vector_size=)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">create the collection if missing</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">self.db.insert_document(name, document, chunks)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">persist document + chunks</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">self.chunk_size / self.overlap / self.strategy / self.collection_name</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">config, already on self</td></tr></table>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Returns</div>\n    <div style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px;color:#1f2933">list[Chunk] — embedded and stored; [] if the document produced no text</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Where it plugs in</div>\n    <div style="font-size:12px;color:#334155">Once written you bind it onto\n      <code style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;color:#0f766e">IngestionCore.ingest_document</code>, and the whole pipeline\n      starts using your version. The cell after it checks your work.</div>\n    <div style="margin-top:10px;padding:7px 10px;background:#fffbeb;border:1px solid #fde68a;border-radius:7px;font-size:12px;color:#78350f">⚠️ Copy the metadata with <code>dict(document.metadata)</code> so chunks don\'t share one mutable dict — and embed in a single batched call, not one call per chunk.</div>\n  </div>\n</div>'))

"chunking.chunk_text(text, chunk_size=, overlap=, strategy=)",the passages to store
"Chunk(document_id=, index=, text=, metadata=)",one chunk object per passage
self.embedder.embed_batch(texts),"one vector per text, order preserved"
"self.db.ensure_collection(name, vector_size=)",create the collection if missing
"self.db.insert_document(name, document, chunks)",persist document + chunks
self.chunk_size / self.overlap / self.strategy / self.collection_name,"config, already on self"


In [25]:
def ingest_document(self, document):
    """Run load->CHUNK->EMBED->store for one Document. Return the list of stored Chunks."""
    # 1. CHUNK — split the text using this core's configured strategy.
    texts = chunking.chunk_text(
        document.text,
        chunk_size=self.chunk_size,
        overlap=self.overlap,
        strategy=self.strategy,
    )
    if not texts:
        return []

    # 2. WRAP — one Chunk per passage, each carrying the document's metadata.
    chunks = [
        Chunk(document_id=document.id, index=i, text=text,
              metadata=dict(document.metadata))                                     # ✏️ a *copy*, so chunks don't share one dict
        for i, text in enumerate(texts)
    ]

    # 3. EMBED — one batched call for the whole document, not one call per chunk.
    embeddings = self.embedder.embed_batch(texts)                                   # ✏️ what gets embedded, in chunk order
    for chunk, embedding in zip(chunks, embeddings):
        chunk.embedding = embedding                                                 # ✏️ attach each vector to its chunk

    # 4. STORE
    self.db.ensure_collection(self.collection_name, vector_size=len(embeddings[0])) # ✏️ how wide is one vector?
    self.db.insert_document(self.collection_name, document, chunks)
    return chunks

In [26]:
IngestionCore.ingest_document = ingest_document
print("Patched IngestionCore.ingest_document")

Patched IngestionCore.ingest_document


In [27]:
suite = TestSuite("Part 1.2 — ingest_document")

def _fresh_core():
    db = DBManager()  # in-memory, ephemeral
    core = IngestionCore(MockEmbedder([1.0, 0.0, 0.0]), db, collection_name="t",
                         chunk_size=5, overlap=1)
    return db, core

_doc = Document(text="one two three four five six seven eight nine ten",
                metadata={"source": "x.md", "document_title": "X"})

@suite.case("ingest_document", "every chunk comes back embedded")
def _():
    _db, core = _fresh_core()
    chunks = core.ingest_document(_doc)
    assert chunks and all(c.embedding is not None for c in chunks)

@suite.case("ingest_document", "chunk indices are sequential from 0")
def _():
    _db, core = _fresh_core()
    chunks = core.ingest_document(_doc)
    assert [c.index for c in chunks] == list(range(len(chunks)))

@suite.case("ingest_document", "chunks are persisted and re-readable from the store")
def _():
    db, core = _fresh_core()
    chunks = core.ingest_document(_doc)
    assert len(db.get_all_chunks("t")) == len(chunks)

@suite.case("ingest_document", "document metadata rides onto every chunk")
def _():
    _db, core = _fresh_core()
    chunks = core.ingest_document(_doc)
    assert all(c.metadata.get("source") == "x.md" for c in chunks)

suite.run()

╭────────────────────────────╮
│ Part 1.2 — ingest_document │
╰────────────────────────────╯

Connected to Qdrant (in-memory mode)
Collection 't' created (vector size: 3)
Inserted 3 chunks into 't'
Connected to Qdrant (in-memory mode)
Collection 't' created (vector size: 3)
Inserted 3 chunks into 't'
Connected to Qdrant (in-memory mode)
Collection 't' created (vector size: 3)
Inserted 3 chunks into 't'
Connected to Qdrant (in-memory mode)
Collection 't' created (vector size: 3)
Inserted 3 chunks into 't'


ingest_document  4/4

✓ every chunk comes back embedded

✓ chunk indices are sequential from 0

✓ chunks are persisted and re-readable from the store

✓ document metadata rides onto every chunk

               Summary               
                                     
  Function          Passed   Status  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  ingest_document    4/4     ✓ PASS

╭─────────────────────────────────────╮
│ All 4 checks passed — nice work! 🎉 │
╰─────────────────────────────────────╯

True

## 1.3 — Metadata

When we load a file we attach **metadata**: where it came from (`source`) and a human title
(`document_title`). That metadata rides on every chunk it produces, is stored alongside the vector,
and is what later lets retrieval *narrow the field* — "only search the leave policy" — so unrelated
documents cannot crowd out the right answer.

Two provided pieces you should read before writing anything. `loaders._derive_title` is where the
title comes from (first Markdown heading, else a tidied file stem), and `metadata_schema` is the
contract every stored chunk must satisfy:

In [28]:
#@title 🏷️ Where titles come from, and what every chunk must carry { display-mode: "form" }
import inspect as _inspect
import metadata_schema

print(_inspect.getsource(loaders._derive_title))
print("Required on every chunk:", sorted(metadata_schema.REQUIRED_FIELDS))
explain("Why validation lives on the write path",
        "<code>DBManager</code> runs <code>validate_payload</code> on every insert. An ingestion "
        "step that forgets to copy the document's metadata onto its chunks fails <b>loudly at "
        "ingestion</b> — instead of silently producing a corpus where every metadata filter matches "
        "nothing and retrieval is mysteriously bad.")

def _derive_title(text: str, stem: str) -> str:
    """First Markdown ``# heading`` if present, else a title-cased file stem."""
    for line in text.splitlines():
        stripped = line.strip()
        if stripped.startswith("# "):
            return stripped[2:].strip()
    return stem.replace("_", " ").replace("-", " ").title()

Required on every chunk: ['document_title', 'source', 'text']


Now the part that *uses* metadata. Every search strategy in Part 2 calls this first, before any
scoring happens: it throws away chunks that don't match, so the ranking only ever sees candidates
from the documents you asked for.

A filter is a plain dict, e.g. `{"source": "leave_and_absence_policy.md"}`. **All** its key/value
pairs must match (AND, not OR), and `None` means "no filter — search everything".

In [29]:
#@title ✏️ Your task · 1.3 { display-mode: "form" }
from IPython.display import HTML, display
display(HTML('<div style="border:1px solid #e3e8ef;border-left:5px solid #0f766e;border-radius:12px;\n     background:linear-gradient(135deg,#fbfefe,#f7fafc);padding:0 0 16px;margin:14px 0;\n     font-family:ui-sans-serif,system-ui,sans-serif;max-width:920px">\n  <div style="background:#0f766e;color:#fff;padding:7px 16px;border-radius:7px 7px 0 0;\n       font-size:11px;font-weight:700;letter-spacing:.09em;text-transform:uppercase">\n    ✏️ Your task &nbsp;·&nbsp; 1.3\n  </div>\n  <div style="padding:0 16px">\n    <div style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:13.5px;color:#1f2933;font-weight:600;\n         background:#fff;border:1px solid #e3e8ef;border-radius:7px;padding:8px 11px;margin-top:13px;\n         overflow-x:auto;white-space:pre">apply_metadata_filter(chunks, metadata_filter)</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">What it must do</div>\n    <div style="font-size:13px;color:#1f2933;line-height:1.55">Keep only the chunks whose metadata matches <b>every</b> key/value pair in the filter (AND, not OR). An empty or <code>None</code> filter means "no filter" — everything passes.</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Use these — already provided</div>\n    <table style="border-collapse:collapse;width:100%"><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">chunk.metadata.get(key)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the chunk\'s value for a key, or None if absent</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">metadata_filter.items()</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the key/value pairs that must all match</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">all(...)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">true only when every pair matches</td></tr></table>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Returns</div>\n    <div style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px;color:#1f2933">list[Chunk] — the surviving candidates, in their original order</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Where it plugs in</div>\n    <div style="font-size:12px;color:#334155">Once written you bind it onto\n      <code style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;color:#0f766e">RetrievalCore._apply_metadata_filter</code>, and the whole pipeline\n      starts using your version. The cell after it checks your work.</div>\n    <div style="margin-top:10px;padding:7px 10px;background:#fffbeb;border:1px solid #fde68a;border-radius:7px;font-size:12px;color:#78350f">⚠️ In the no-filter case return the <b>same list object</b> you were given, not a copy: the BM25 cache uses that identity to recognise the full corpus.</div>\n  </div>\n</div>'))

chunk.metadata.get(key),"the chunk's value for a key, or None if absent"
metadata_filter.items(),the key/value pairs that must all match
all(...),true only when every pair matches


In [30]:
def apply_metadata_filter(chunks, metadata_filter):
    """Keep only chunks whose metadata matches every key/value in the filter."""
    if not metadata_filter:
        return chunks                                                                       # ✏️ no filter: hand back the *same list object* (the BM25 cache
                                                                                            #    recognises the full corpus by identity, so don't copy it)
    return [
        chunk for chunk in chunks
        if all(chunk.metadata.get(key) == value for key, value in metadata_filter.items())  # ✏️ what every pair must satisfy
    ]

In [31]:
# _apply_metadata_filter is a @staticmethod, so wrap it when patching.
RetrievalCore._apply_metadata_filter = staticmethod(apply_metadata_filter)
print("Patched RetrievalCore._apply_metadata_filter")

Patched RetrievalCore._apply_metadata_filter


In [32]:
suite = TestSuite("Part 1.3 — apply_metadata_filter")

_a = Chunk(document_id="d", index=0, text="A", metadata={"source": "a.md", "year": 2024})
_b = Chunk(document_id="d", index=1, text="B", metadata={"source": "b.md", "year": 2024})

@suite.case("apply_metadata_filter", "keeps only the matching chunks")
def _():
    assert apply_metadata_filter([_a, _b], {"source": "a.md"}) == [_a]

@suite.case("apply_metadata_filter", "None means no filter — everything passes")
def _():
    assert apply_metadata_filter([_a, _b], None) == [_a, _b]

@suite.case("apply_metadata_filter", "an empty filter also means no filter")
def _():
    assert apply_metadata_filter([_a, _b], {}) == [_a, _b]

@suite.case("apply_metadata_filter", "every key must match (AND, not OR)")
def _():
    assert apply_metadata_filter([_a, _b], {"source": "a.md", "year": 2024}) == [_a]
    assert apply_metadata_filter([_a, _b], {"source": "a.md", "year": 1999}) == []

@suite.case("apply_metadata_filter", "an unknown key matches nothing rather than raising")
def _():
    assert apply_metadata_filter([_a, _b], {"department": "HR"}) == []

suite.run()

╭──────────────────────────────────╮
│ Part 1.3 — apply_metadata_filter │
╰──────────────────────────────────╯

apply_metadata_filter  5/5

✓ keeps only the matching chunks

✓ None means no filter — everything passes

✓ an empty filter also means no filter

✓ every key must match (AND, not OR)

✓ an unknown key matches nothing rather than raising

                  Summary                  
                                           
  Function                Passed   Status  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  apply_metadata_filter    5/5     ✓ PASS

╭─────────────────────────────────────╮
│ All 5 checks passed — nice work! 🎉 │
╰─────────────────────────────────────╯

True

**The payoff.** Ingest all nine policies, then watch the filter cut the search space before a single
score is computed:

In [33]:
#@title 🎯 A metadata filter, before any scoring happens { display-mode: "form" }
all_chunks = []
_db = DBManager()
_core = IngestionCore(HashingEmbedder(), _db, collection_name="demo",
                      chunk_size=120, overlap=20)
for d in docs:
    all_chunks.extend(_core.ingest_document(d))

only_leave = RetrievalCore._apply_metadata_filter(all_chunks, {"source": "leave_and_absence_policy.md"})
display(HTML(_card(
    f'<b style="color:{_INK}">Metadata filter</b> '
    f'<code style="color:{_TEAL}">{{"source": "leave_and_absence_policy.md"}}</code><br>'
    f'corpus: <b>{len(all_chunks)}</b> chunks &nbsp;→&nbsp; '
    f'after filter: <b style="color:{_TEAL}">{len(only_leave)}</b> chunks')))
show_chunks(only_leave, title="Leave-policy chunks (after filtering)")

explain("Filtering is not ranking",
        "The filter runs <b>before</b> scoring and is a hard yes/no on metadata — it cannot rank, "
        "and it cannot rescue a query whose wording misses. Getting the right document into the "
        "candidate pool is metadata's job; ordering what is in the pool is Part 2's job.",
        tone="info", icon="🎯")

Connected to Qdrant (in-memory mode)
Collection 'demo' created (vector size: 128)
Inserted 10 chunks into 'demo'
Inserted 13 chunks into 'demo'
Inserted 2 chunks into 'demo'
Inserted 10 chunks into 'demo'
Inserted 12 chunks into 'demo'
Inserted 11 chunks into 'demo'
Inserted 13 chunks into 'demo'
Inserted 11 chunks into 'demo'
Inserted 2 chunks into 'demo'


## 1.4 — Export your Part 1 functions

This writes your three functions to `solutions/part1_ingestion.py`. Dropped into `project/`, the app
will run on *your* ingestion code (see Part 3). **Run your implementation cells above first**, then
run this.

In [35]:
HEADER_P1 = (
    '"""Part 1 — exported from the notebook. Do not edit by hand; re-export instead."""\n'
    'from __future__ import annotations\n'
    'import chunking\n'
    'from models import Chunk\n\n\n'
)

_funcs_p1 = [sliding_window, ingest_document, apply_metadata_filter]

# A function that still contains a TODO is not exported: the app would import it and
# crash. Left out, that function simply stays unimplemented in the app — there is no
# reference version of it in the repo to fall back on.
_ready_p1 = [f for f in _funcs_p1 if not has_blanks(f)]
for f in _funcs_p1:
    if has_blanks(f):
        print(f"⚠️  Skipping {f.__name__} — it still has unfilled TODO blanks.")

_body = HEADER_P1 + "\n\n".join(inspect.getsource(f) for f in _ready_p1)

_path = pathlib.Path("solutions/part1_ingestion.py")
_path.parent.mkdir(exist_ok=True)
_path.write_text(_body, encoding="utf-8")
print(f"Wrote {_path} ({len(_ready_p1)}/{len(_funcs_p1)} functions).")

try:
    from google.colab import files
    files.download(str(_path))
except Exception:
    pass

Wrote solutions\part1_ingestion.py (3/3 functions).


### Bonus (optional): smarter vector stores

The provided store keeps chunks in a flat list and compares the query against *every* vector.
That's fine for a few thousand chunks but doesn't scale. Real systems use **approximate
nearest-neighbour** indexes such as **HNSW** (Hierarchical Navigable Small World graphs), which
trade a tiny bit of accuracy for enormous speed. Qdrant (the store under `DBManager`) uses HNSW
internally. *Bonus:* read how HNSW builds a layered graph and why it's so much faster than a linear
scan.

---
# Part 2 — The RAG core

Now the **read side**: take a question and produce a grounded answer.

In [36]:
#@title 🔎 The retrieval path { display-mode: "form" }
from IPython.display import HTML, display
display(HTML('<div style="text-align:center;margin:10px 0"><svg viewBox="0 0 900 214" width="100%" style="max-width:900px;font-family:ui-sans-serif,system-ui,sans-serif"><defs><marker id="ah" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L8,3 L0,6 Z" fill="#647084"/></marker></defs><rect x="16" y="82" width="124" height="54" rx="10" fill="#ffffff" stroke="#334155" stroke-width="1.6"/><text x="78.0" y="114.0" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Question</text><rect x="190" y="18" width="168" height="54" rx="10" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/><text x="274.0" y="41" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Keyword search</text><text x="274.0" y="58" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">BM25  ✏️</text><rect x="190" y="138" width="168" height="54" rx="10" fill="#ffffff" stroke="#0f766e" stroke-width="1.6"/><text x="274.0" y="161" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Embedding search</text><text x="274.0" y="178" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">cosine  ✏️</text><rect x="400" y="82" width="116" height="54" rx="10" fill="#ffffff" stroke="#b45309" stroke-width="1.6"/><text x="458.0" y="105" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Fuse</text><text x="458.0" y="122" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">RRF  ✏️</text><rect x="548" y="82" width="116" height="54" rx="10" fill="#ffffff" stroke="#e3e8ef" stroke-width="1.6"/><text x="606.0" y="105" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Rerank</text><text x="606.0" y="122" text-anchor="middle" font-size="10.5" font-weight="400" fill="#647084">bonus</text><rect x="696" y="82" width="92" height="54" rx="10" fill="#ffffff" stroke="#334155" stroke-width="1.6"/><text x="742.0" y="114.0" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">LLM</text><rect x="800" y="82" width="92" height="54" rx="10" fill="#ffffff" stroke="#b45309" stroke-width="1.6"/><text x="846.0" y="114.0" text-anchor="middle" font-size="13.5" font-weight="700" fill="#1f2933">Answer</text><path d="M140,100 L190,58" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M140,114 L190,178" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M358,58 L400,100" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M358,178 L400,120" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M516,109 L548,109" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M664,109 L696,109" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/><path d="M788,109 L800,109" stroke="#647084" stroke-width="1.6" fill="none" marker-end="url(#ah)"/></svg></div>'))

You'll build each retrieval strategy (the ✏️ steps), fuse them, **measure** all of them against a
fixed set of questions, and finally hand the chosen chunks to the LLM.

## 2.0 — First, watch retrieval fail

Before building anything, see the problem. Here are two questions with the same answer document.
One uses the policy's own vocabulary; the other is phrased the way a person would actually ask.

In [37]:
#@title 🔍 The same question asked two ways — watch BM25 fail { display-mode: "form" }
# A throwaway BM25 ranking, spelled out here because you don't build the real one
# until §2.2 — this cell has to run before that.
def _bm25_demo(query, chunks, top_k=3):
    index = BM25Okapi([_tokenize(c.text) for c in chunks])
    scores = index.get_scores(_tokenize(query))
    order = sorted(range(len(chunks)), key=lambda i: scores[i], reverse=True)[:top_k]
    return [(chunks[i], float(scores[i])) for i in order]

_gold = "leave_and_absence_policy.md"

show_compare([
    ('Asked in the policy\'s words:<br><code>"annual leave entitlement days"</code>',
     _bm25_demo("annual leave entitlement days", all_chunks), _gold),
    ('Asked the way a person asks:<br><code>"how much time off do I get each year?"</code>',
     _bm25_demo("how much time off do I get each year?", all_chunks), _gold),
], note="Keyword (BM25) search. Green = the document that actually answers the question.")

Same question, different words, and the second one falls apart. BM25 scores **exact token
overlap** — "time off" and "annual leave" share no tokens at all, so the document that holds the
answer is invisible to it. Nothing is broken; this is what lexical matching *is*.

That single failure mode is why the rest of Part 2 exists:

In [38]:
#@title 🧭 What you are about to build { display-mode: "form" }
explain("The three ideas you are about to build",
        "<b>1. Embedding search</b> ranks by meaning rather than words, so it survives the "
        "vocabulary mismatch above — but it is fuzzy, and can miss an exact term like an acronym "
        "or an article number.<br>"
        "<b>2. Hybrid search</b> runs both and fuses the rankings, so a question only has to be "
        "answerable by <i>one</i> of them.<br>"
        "<b>3. Measurement</b> (§2.5) is how you know any of that is true instead of plausible.",
        icon="🧭")

## 2.1 — Embedding (semantic) search

Semantic search ranks chunks by how close their vectors are to the **query's** vector. The standard
closeness measure is **cosine similarity** — the angle between two vectors, ignoring their length:

```
cos(q, d) = (q · d) / (||q|| * ||d||)
```

In [39]:
#@title 📐 Cosine similarity, geometrically { display-mode: "form" }
from IPython.display import HTML, display
display(HTML('<div style="text-align:center;margin:10px 0"><svg viewBox="0 0 520 210" width="100%" style="max-width:520px;font-family:ui-sans-serif,system-ui,sans-serif"><defs><marker id="ah" markerWidth="9" markerHeight="9" refX="7" refY="3" orient="auto"><path d="M0,0 L8,3 L0,6 Z" fill="#647084"/></marker></defs><line x1="60" y1="170" x2="250" y2="50" stroke="#0f766e" stroke-width="2.4" marker-end="url(#ah)"/><line x1="60" y1="170" x2="270" y2="120" stroke="#b45309" stroke-width="2.4" marker-end="url(#ah)"/><path d="M120,135 A 62 62 0 0 1 128 116" fill="none" stroke="#647084" stroke-width="1.4"/><text x="150" y="132" text-anchor="middle" font-size="14" font-weight="700" fill="#1f2933">θ</text><text x="255" y="42" text-anchor="start" font-size="12" font-weight="700" fill="#0f766e">query vector q</text><text x="276" y="122" text-anchor="start" font-size="12" font-weight="700" fill="#b45309">chunk vector d</text><text x="60" y="190" text-anchor="start" font-size="11" font-weight="600" fill="#334155">cos(q, d) = (q · d) / (‖q‖ ‖d‖)  —  small angle → score near 1</text></svg></div>'))

You'll implement it in a vectorised way: `query_vec` is one vector, `matrix` has one chunk vector
per row, and you return one score per row. Guard against divide-by-zero (a zero vector → score 0).

In [40]:
#@title ✏️ Your task · 2.1 { display-mode: "form" }
from IPython.display import HTML, display
display(HTML('<div style="border:1px solid #e3e8ef;border-left:5px solid #0f766e;border-radius:12px;\n     background:linear-gradient(135deg,#fbfefe,#f7fafc);padding:0 0 16px;margin:14px 0;\n     font-family:ui-sans-serif,system-ui,sans-serif;max-width:920px">\n  <div style="background:#0f766e;color:#fff;padding:7px 16px;border-radius:7px 7px 0 0;\n       font-size:11px;font-weight:700;letter-spacing:.09em;text-transform:uppercase">\n    ✏️ Your task &nbsp;·&nbsp; 2.1\n  </div>\n  <div style="padding:0 16px">\n    <div style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:13.5px;color:#1f2933;font-weight:600;\n         background:#fff;border:1px solid #e3e8ef;border-radius:7px;padding:8px 11px;margin-top:13px;\n         overflow-x:auto;white-space:pre">cosine_similarity(query_vec, matrix)</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">What it must do</div>\n    <div style="font-size:13px;color:#1f2933;line-height:1.55">Score one query vector against every chunk vector at once. Cosine similarity is the dot product divided by the product of the norms — do it <b>vectorised</b>, one score per row of <code>matrix</code>, not in a Python loop.</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Use these — already provided</div>\n    <table style="border-collapse:collapse;width:100%"><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">matrix @ query_vec</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the dot product of every row with the query</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">np.linalg.norm(matrix, axis=1)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the norm of every row</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">np.linalg.norm(query_vec)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the query\'s norm</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">np.divide(a, b, out=, where=)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">divide safely where the denominator is non-zero</td></tr></table>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Returns</div>\n    <div style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px;color:#1f2933">np.ndarray of shape (N,) — one score per row of matrix</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Where it plugs in</div>\n    <div style="font-size:12px;color:#334155">Once written you bind it onto\n      <code style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;color:#0f766e">RetrievalCore._cosine_similarity</code>, and the whole pipeline\n      starts using your version. The cell after it checks your work.</div>\n    <div style="margin-top:10px;padding:7px 10px;background:#fffbeb;border:1px solid #fde68a;border-radius:7px;font-size:12px;color:#78350f">⚠️ A zero vector has zero norm. Divide naively and you get NaN, which silently poisons every ranking downstream — it must score 0.0 instead.</div>\n  </div>\n</div>'))

matrix @ query_vec,the dot product of every row with the query
"np.linalg.norm(matrix, axis=1)",the norm of every row
np.linalg.norm(query_vec),the query's norm
"np.divide(a, b, out=, where=)",divide safely where the denominator is non-zero


In [41]:
def cosine_similarity(query_vec, matrix):
    """Cosine similarity of `query_vec` against every row of `matrix`."""
    # query_vec: (D,)   matrix: (N, D)   returns: (N,)
    # The denominator ||row|| * ||query||, one value per row of the matrix.
    denom = np.linalg.norm(matrix, axis=1) * np.linalg.norm(query_vec)              # ✏️ which axis is "per row"?
    # The numerator: the dot product of every row with the query.
    scores = matrix @ query_vec   # ✏️ every row against the query in one operation — no Python loop
    # Divide only where the denominator is non-zero: a zero vector scores 0.0, never NaN.
    return np.divide(scores, denom, out=np.zeros_like(scores), where=denom != 0)    # ✏️ the safety condition

In [42]:
# _cosine_similarity is a @staticmethod, so wrap it when patching.
RetrievalCore._cosine_similarity = staticmethod(cosine_similarity)
print("Patched RetrievalCore._cosine_similarity")

Patched RetrievalCore._cosine_similarity


In [43]:
suite = TestSuite("Part 2.1 — cosine_similarity")

@suite.case("cosine_similarity", "identical direction -> 1.0")
def _():
    q = np.array([1.0, 0.0, 0.0], dtype=np.float32)
    M = np.array([[1, 0, 0], [2, 0, 0]], dtype=np.float32)
    out = cosine_similarity(q, M)
    assert np.allclose(out, [1.0, 1.0])

@suite.case("cosine_similarity", "orthogonal -> 0.0")
def _():
    q = np.array([1.0, 0.0, 0.0], dtype=np.float32)
    M = np.array([[0, 1, 0]], dtype=np.float32)
    assert np.isclose(cosine_similarity(q, M)[0], 0.0)

@suite.case("cosine_similarity", "zero vector is safe (no NaN)")
def _():
    q = np.array([1.0, 0.0, 0.0], dtype=np.float32)
    M = np.array([[0, 0, 0]], dtype=np.float32)
    assert cosine_similarity(q, M)[0] == 0.0

suite.run()

╭──────────────────────────────╮
│ Part 2.1 — cosine_similarity │
╰──────────────────────────────╯

cosine_similarity  3/3

✓ identical direction -> 1.0

✓ orthogonal -> 0.0

✓ zero vector is safe (no NaN)

                Summary                
                                       
  Function            Passed   Status  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  cosine_similarity    3/3     ✓ PASS

╭─────────────────────────────────────╮
│ All 3 checks passed — nice work! 🎉 │
╰─────────────────────────────────────╯

True

## 2.2 — Keyword (lexical) search

Embeddings capture meaning but can miss an exact term — a specific article number, an acronym, a
rare word. **Keyword search** complements them by scoring exact token overlap. We use **BM25**, a
classic ranking function that rewards query terms appearing in a chunk while down-weighting words
that are common across the whole corpus.

The helpers are provided: `_tokenize(text)` (lowercase + split), `self._apply_metadata_filter(...)`
(the one you wrote in 1.3), `self._top_k(candidates, scores, top_k)`, and `self._bm25_index(candidates)`.

That last one deserves a word. The obvious line is:

```python
bm25 = BM25Okapi([_tokenize(c.text) for c in candidates])   # rebuilt on EVERY query
```

BM25 needs the whole corpus to compute its IDF term, so building the index is O(corpus) — and that
line pays it on every single query, twice per hybrid query, over a wide over-retrieval pool.
`self._bm25_index(candidates)` builds the same index but **caches** it for the unfiltered corpus,
invalidated whenever `self.chunks` is replaced. Same scores, one build per ingest instead of one per
query.

In [44]:
#@title ✏️ Your task · 2.2 { display-mode: "form" }
from IPython.display import HTML, display
display(HTML('<div style="border:1px solid #e3e8ef;border-left:5px solid #0f766e;border-radius:12px;\n     background:linear-gradient(135deg,#fbfefe,#f7fafc);padding:0 0 16px;margin:14px 0;\n     font-family:ui-sans-serif,system-ui,sans-serif;max-width:920px">\n  <div style="background:#0f766e;color:#fff;padding:7px 16px;border-radius:7px 7px 0 0;\n       font-size:11px;font-weight:700;letter-spacing:.09em;text-transform:uppercase">\n    ✏️ Your task &nbsp;·&nbsp; 2.2\n  </div>\n  <div style="padding:0 16px">\n    <div style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:13.5px;color:#1f2933;font-weight:600;\n         background:#fff;border:1px solid #e3e8ef;border-radius:7px;padding:8px 11px;margin-top:13px;\n         overflow-x:auto;white-space:pre">keyword_search(self, query, top_k=None, metadata_filter=None)</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">What it must do</div>\n    <div style="font-size:13px;color:#1f2933;line-height:1.55">Rank the corpus by BM25 overlap with the query. Narrow the candidates by metadata first, score what is left, and return the best <code>top_k</code> as (chunk, score) pairs.</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Use these — already provided</div>\n    <table style="border-collapse:collapse;width:100%"><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">self.top_k if top_k is None else top_k</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">fall back to the configured default</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">_tokenize(text)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">lowercase + split into word tokens</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">self._apply_metadata_filter(self.chunks, metadata_filter)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the candidates to score</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">self._bm25_index(candidates)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">a BM25 index, cached for the full corpus</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">bm25.get_scores(query_tokens)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">one score per candidate</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">self._top_k(candidates, scores, top_k)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the best k, as (chunk, score) pairs</td></tr></table>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Returns</div>\n    <div style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px;color:#1f2933">list[tuple[Chunk, float]] — highest score first</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Where it plugs in</div>\n    <div style="font-size:12px;color:#334155">Once written you bind it onto\n      <code style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;color:#0f766e">RetrievalCore.keyword_search</code>, and the whole pipeline\n      starts using your version. The cell after it checks your work.</div>\n    <div style="margin-top:10px;padding:7px 10px;background:#fffbeb;border:1px solid #fde68a;border-radius:7px;font-size:12px;color:#78350f">⚠️ Return <code>[]</code> early when there are no candidates or no query tokens — BM25 raises on an empty corpus.</div>\n  </div>\n</div>'))

self.top_k if top_k is None else top_k,fall back to the configured default
_tokenize(text),lowercase + split into word tokens
"self._apply_metadata_filter(self.chunks, metadata_filter)",the candidates to score
self._bm25_index(candidates),"a BM25 index, cached for the full corpus"
bm25.get_scores(query_tokens),one score per candidate
"self._top_k(candidates, scores, top_k)","the best k, as (chunk, score) pairs"


In [46]:
def keyword_search(self, query, top_k=None, metadata_filter=None):
    """Rank self.chunks by BM25 overlap with `query`. Return list[(Chunk, score)]."""
    top_k = self.top_k if top_k is None else top_k
    query_tokens = _tokenize(query)

    candidates = self._apply_metadata_filter(self.chunks, metadata_filter)  # ✏️ narrow the corpus by metadata *before* scoring
    if not candidates or not query_tokens:
        return []                                                           # BM25 raises on an empty corpus

    bm25 = self._bm25_index(candidates)                                     # cached for the unfiltered corpus
    scores = bm25.get_scores(query_tokens)                                  # ✏️ one BM25 score per candidate
    return self._top_k(candidates, scores, top_k)

In [47]:
RetrievalCore.keyword_search = keyword_search
print("Patched RetrievalCore.keyword_search")

Patched RetrievalCore.keyword_search


In [48]:
suite = TestSuite("Part 2.2 — keyword_search")

_kw_chunks = [
    Chunk(document_id="d", index=0, text="employees get twenty five days of annual leave",
          metadata={"source": "leave.md"}),
    Chunk(document_id="d", index=1, text="expenses must be submitted within thirty days",
          metadata={"source": "expense.md"}),
]
_rc = RetrievalCore(MockEmbedder(), chunks=_kw_chunks)

@suite.case("keyword_search", "the chunk with the query terms ranks first")
def _():
    res = _rc.keyword_search("how many days of annual leave", top_k=2)
    assert res[0][0].index == 0

@suite.case("keyword_search", "top_k caps the number of results")
def _():
    assert len(_rc.keyword_search("days", top_k=1)) == 1

@suite.case("keyword_search", "a metadata filter restricts the candidates")
def _():
    res = _rc.keyword_search("days", metadata_filter={"source": "expense.md"})
    assert all(c.metadata["source"] == "expense.md" for c, _ in res)

@suite.case("keyword_search", "empty query -> no results")
def _():
    assert _rc.keyword_search("   ") == []

suite.run()

╭───────────────────────────╮
│ Part 2.2 — keyword_search │
╰───────────────────────────╯

keyword_search  4/4

✓ the chunk with the query terms ranks first

✓ top_k caps the number of results

✓ a metadata filter restricts the candidates

✓ empty query -> no results

              Summary               
                                    
  Function         Passed   Status  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  keyword_search    4/4     ✓ PASS

╭─────────────────────────────────────╮
│ All 4 checks passed — nice work! 🎉 │
╰─────────────────────────────────────╯

True

## 2.3 — Combining the two: Reciprocal Rank Fusion

Keyword and embedding search return scores on **incompatible scales** (BM25 vs. cosine), so you
can't just add them. **Reciprocal Rank Fusion (RRF)** sidesteps this by using only each result's
*rank*:

```
fused_score(chunk) = Σ over lists  1 / (k + rank_in_that_list)        # rank is 0-based, k = 60
```

A chunk that ranks high in *either* list scores well; a chunk near the top of *both* wins. `k=60`
is the conventional dampening constant. Dedupe by `chunk.id` (a chunk can appear in both lists).

In [49]:
#@title 🔀 Reciprocal Rank Fusion, step by step { display-mode: "form" }
from IPython.display import HTML, display
display(HTML('<div style="display:flex;gap:14px;align-items:center;justify-content:center;flex-wrap:wrap;margin:12px 0;font-family:ui-sans-serif,system-ui,sans-serif"><div style="border:1px solid #e3e8ef;border-top:3px solid #0f766e;border-radius:8px;padding:8px 12px;min-width:130px"><div style="font-size:11px;color:#0f766e;font-weight:700;text-transform:uppercase;letter-spacing:.04em;margin-bottom:2px">keyword</div><div style="display:flex;gap:8px;padding:3px 0"><span style="color:#647084;font-family:ui-monospace,monospace;font-size:11px">#1</span><span style="color:#1f2933;font-weight:600;font-size:13px">chunk A</span></div><div style="display:flex;gap:8px;padding:3px 0"><span style="color:#647084;font-family:ui-monospace,monospace;font-size:11px">#2</span><span style="color:#1f2933;font-weight:600;font-size:13px">chunk B</span></div></div><div style="border:1px solid #e3e8ef;border-top:3px solid #0f766e;border-radius:8px;padding:8px 12px;min-width:130px"><div style="font-size:11px;color:#0f766e;font-weight:700;text-transform:uppercase;letter-spacing:.04em;margin-bottom:2px">embedding</div><div style="display:flex;gap:8px;padding:3px 0"><span style="color:#647084;font-family:ui-monospace,monospace;font-size:11px">#1</span><span style="color:#1f2933;font-weight:600;font-size:13px">chunk B</span></div><div style="display:flex;gap:8px;padding:3px 0"><span style="color:#647084;font-family:ui-monospace,monospace;font-size:11px">#2</span><span style="color:#1f2933;font-weight:600;font-size:13px">chunk C</span></div></div><div style="font-size:24px;color:#647084">→</div><div style="border:1px solid #e3e8ef;border-top:3px solid #b45309;border-radius:8px;padding:8px 12px;min-width:130px"><div style="font-size:11px;color:#b45309;font-weight:700;text-transform:uppercase;letter-spacing:.04em;margin-bottom:2px">fused (RRF)</div><div style="display:flex;gap:8px;padding:3px 0"><span style="color:#647084;font-family:ui-monospace,monospace;font-size:11px">#1</span><span style="color:#1f2933;font-weight:600;font-size:13px">chunk B · 1/60 + 1/61</span></div><div style="display:flex;gap:8px;padding:3px 0"><span style="color:#647084;font-family:ui-monospace,monospace;font-size:11px">#2</span><span style="color:#1f2933;font-weight:600;font-size:13px">chunk A · 1/60</span></div><div style="display:flex;gap:8px;padding:3px 0"><span style="color:#647084;font-family:ui-monospace,monospace;font-size:11px">#3</span><span style="color:#1f2933;font-weight:600;font-size:13px">chunk C · 1/61</span></div></div></div>'))

Above: **chunk B** isn't first in either list, but it ranks well in *both*, so fusion floats it to
the top — exactly the behaviour we want.

In [50]:
#@title ✏️ Your task · 2.3 { display-mode: "form" }
from IPython.display import HTML, display
display(HTML('<div style="border:1px solid #e3e8ef;border-left:5px solid #0f766e;border-radius:12px;\n     background:linear-gradient(135deg,#fbfefe,#f7fafc);padding:0 0 16px;margin:14px 0;\n     font-family:ui-sans-serif,system-ui,sans-serif;max-width:920px">\n  <div style="background:#0f766e;color:#fff;padding:7px 16px;border-radius:7px 7px 0 0;\n       font-size:11px;font-weight:700;letter-spacing:.09em;text-transform:uppercase">\n    ✏️ Your task &nbsp;·&nbsp; 2.3\n  </div>\n  <div style="padding:0 16px">\n    <div style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:13.5px;color:#1f2933;font-weight:600;\n         background:#fff;border:1px solid #e3e8ef;border-radius:7px;padding:8px 11px;margin-top:13px;\n         overflow-x:auto;white-space:pre">reciprocal_rank_fusion(ranked_lists, top_k, k=60)</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">What it must do</div>\n    <div style="font-size:13px;color:#1f2933;line-height:1.55">Merge several ranked lists into one using only each chunk\'s <b>rank</b>, never its score. Every list a chunk appears in contributes <code>1 / (k + rank)</code> to its total, so a chunk that both retrievers ranked well beats one that only a single retriever loved.</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Use these — already provided</div>\n    <table style="border-collapse:collapse;width:100%"><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">enumerate(ranked)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the 0-based rank of each (chunk, score) pair</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">fused[chunk.id] = fused.get(chunk.id, 0.0) + ...</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">accumulate across lists</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">chunks_by_id[chunk.id] = chunk</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">keep the object so you can return it</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">sorted(..., key=..., reverse=True)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">best fused score first</td></tr></table>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Returns</div>\n    <div style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px;color:#1f2933">list[tuple[Chunk, float]] — at most top_k pairs, deduplicated by chunk id</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Where it plugs in</div>\n    <div style="font-size:12px;color:#334155">Once written you bind it onto\n      <code style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;color:#0f766e">RetrievalCore._reciprocal_rank_fusion</code>, and the whole pipeline\n      starts using your version. The cell after it checks your work.</div>\n    <div style="margin-top:10px;padding:7px 10px;background:#fffbeb;border:1px solid #fde68a;border-radius:7px;font-size:12px;color:#78350f">⚠️ Key everything by <code>chunk.id</code>. The same chunk usually appears in both input lists, and it must come back once — with its scores added, not listed twice.</div>\n  </div>\n</div>'))

enumerate(ranked),"the 0-based rank of each (chunk, score) pair"
"fused[chunk.id] = fused.get(chunk.id, 0.0) + ...",accumulate across lists
chunks_by_id[chunk.id] = chunk,keep the object so you can return it
"sorted(..., key=..., reverse=True)",best fused score first


In [51]:
def reciprocal_rank_fusion(ranked_lists, top_k, k=60):
    """Fuse several ranked [(Chunk, score)] lists into one. Return list[(Chunk, fused_score)]."""
    fused_scores = {}   # chunk id -> summed contribution across every list
    chunks_by_id = {}   # chunk id -> the Chunk itself, so we can return objects
    for ranked in ranked_lists:
        for rank, (chunk, _score) in enumerate(ranked):   # rank is 0-based
            # Rank, never score: each list a chunk appears in adds 1 / (k + rank).
            fused_scores[chunk.id] = fused_scores.get(chunk.id, 0.0) + 1 / (k + rank)   # ✏️ this list's contribution
            chunks_by_id[chunk.id] = chunk

    ordered = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)            # ✏️ sort by fused score, not by id
    return [(chunks_by_id[cid], score) for cid, score in ordered[:top_k]]

In [52]:
RetrievalCore._reciprocal_rank_fusion = staticmethod(reciprocal_rank_fusion)
print("Patched RetrievalCore._reciprocal_rank_fusion")

Patched RetrievalCore._reciprocal_rank_fusion


In [53]:
suite = TestSuite("Part 2.3 — reciprocal_rank_fusion")

_a = Chunk(document_id="d", index=0, text="A")
_b = Chunk(document_id="d", index=1, text="B")
_c = Chunk(document_id="d", index=2, text="C")

@suite.case("reciprocal_rank_fusion", "a chunk ranked highly in both lists wins")
def _():
    list1 = [(_a, 9.0), (_b, 1.0)]   # b at rank 1
    list2 = [(_b, 9.0), (_c, 1.0)]   # b at rank 0  -> b appears in both
    fused = reciprocal_rank_fusion([list1, list2], top_k=3)
    assert fused[0][0].id == _b.id

@suite.case("reciprocal_rank_fusion", "results are deduplicated by chunk id")
def _():
    fused = reciprocal_rank_fusion([[(_a, 1.0)], [(_a, 1.0)]], top_k=5)
    assert len(fused) == 1

@suite.case("reciprocal_rank_fusion", "top_k limits the output")
def _():
    fused = reciprocal_rank_fusion([[(_a, 1.0), (_b, 1.0), (_c, 1.0)]], top_k=2)
    assert len(fused) == 2

suite.run()

╭───────────────────────────────────╮
│ Part 2.3 — reciprocal_rank_fusion │
╰───────────────────────────────────╯

reciprocal_rank_fusion  3/3

✓ a chunk ranked highly in both lists wins

✓ results are deduplicated by chunk id

✓ top_k limits the output

                  Summary                   
                                            
  Function                 Passed   Status  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  reciprocal_rank_fusion    3/3     ✓ PASS

╭─────────────────────────────────────╮
│ All 3 checks passed — nice work! 🎉 │
╰─────────────────────────────────────╯

True

With all three patched in, the pieces of **hybrid search** are in place. One question remains, and
it matters more than it looks: *how many candidates do you hand the fuser?*

## 2.4 — Hybrid search: over-retrieve, then fuse

Fusion can only rerank what it is given. If each retriever returns just `top_k`, the fuser sees at
most `2 * top_k` candidates — and a chunk sitting 8th in the BM25 list and 6th in the cosine list,
*decent in both* and exactly what RRF exists to promote, can never surface, because neither list was
long enough to contain it.

The fix is **over-retrieval**: pull `top_k * overretrieve` candidates from each branch, fuse that
wide pool, then cut to `top_k`. Here it is close to free — both branches already score *every* chunk
(BM25 over the whole corpus, cosine over the whole matrix) and then throw the rest away, so a wider
pool changes how many results you keep, not how much work you do.

> **Reflection:** `overretrieve=1` is the narrow behaviour described above. What happens once
> `top_k * overretrieve` grows past the size of the corpus?

In [54]:
#@title ✏️ Your task · 2.4 { display-mode: "form" }
from IPython.display import HTML, display
display(HTML('<div style="border:1px solid #e3e8ef;border-left:5px solid #0f766e;border-radius:12px;\n     background:linear-gradient(135deg,#fbfefe,#f7fafc);padding:0 0 16px;margin:14px 0;\n     font-family:ui-sans-serif,system-ui,sans-serif;max-width:920px">\n  <div style="background:#0f766e;color:#fff;padding:7px 16px;border-radius:7px 7px 0 0;\n       font-size:11px;font-weight:700;letter-spacing:.09em;text-transform:uppercase">\n    ✏️ Your task &nbsp;·&nbsp; 2.4\n  </div>\n  <div style="padding:0 16px">\n    <div style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:13.5px;color:#1f2933;font-weight:600;\n         background:#fff;border:1px solid #e3e8ef;border-radius:7px;padding:8px 11px;margin-top:13px;\n         overflow-x:auto;white-space:pre">hybrid_search(self, query, top_k=None, metadata_filter=None, overretrieve=None)</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">What it must do</div>\n    <div style="font-size:13px;color:#1f2933;line-height:1.55">Run both retrievers <b>wide</b>, fuse their rankings, then cut to <code>top_k</code>. Each branch returns <code>top_k × overretrieve</code> candidates so fusion has a real pool to work with — it can only reorder what you hand it.</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Use these — already provided</div>\n    <table style="border-collapse:collapse;width:100%"><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">self.top_k / self.overretrieve / self.rrf_k</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the configured defaults on self</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">max(top_k, top_k * overretrieve)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">how many candidates each branch should return</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">self.keyword_search(query, top_k=pool, metadata_filter=)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the lexical ranking</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">self.embedding_search(query, top_k=pool, metadata_filter=)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the semantic ranking</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">self._reciprocal_rank_fusion([kw, emb], top_k=, k=self.rrf_k)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the fused result</td></tr></table>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Returns</div>\n    <div style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px;color:#1f2933">list[tuple[Chunk, float]] — top_k pairs scored by fused rank</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Where it plugs in</div>\n    <div style="font-size:12px;color:#334155">Once written you bind it onto\n      <code style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;color:#0f766e">RetrievalCore.hybrid_search</code>, and the whole pipeline\n      starts using your version. The cell after it checks your work.</div>\n    <div style="margin-top:10px;padding:7px 10px;background:#fffbeb;border:1px solid #fde68a;border-radius:7px;font-size:12px;color:#78350f">⚠️ Pass <code>metadata_filter</code> through to <b>both</b> branches. Filtering one and not the other lets excluded documents back in through the side door.</div>\n  </div>\n</div>'))

self.top_k / self.overretrieve / self.rrf_k,the configured defaults on self
"max(top_k, top_k * overretrieve)",how many candidates each branch should return
"self.keyword_search(query, top_k=pool, metadata_filter=)",the lexical ranking
"self.embedding_search(query, top_k=pool, metadata_filter=)",the semantic ranking
"self._reciprocal_rank_fusion([kw, emb], top_k=, k=self.rrf_k)",the fused result


In [55]:
def hybrid_search(self, query, top_k=None, metadata_filter=None, overretrieve=None):
    """Run both retrievers wide, fuse their rankings, then cut to top_k."""
    top_k = self.top_k if top_k is None else top_k
    overretrieve = self.overretrieve if overretrieve is None else overretrieve

    # Over-retrieve: fusion can only reorder what you hand it.
    pool = top_k * overretrieve  # ✏️ candidates per branch — never fewer than top_k

    keyword_results = self.keyword_search(query, top_k=pool, metadata_filter=metadata_filter)
    # ✏️ the semantic branch gets exactly the same treatment — same pool, same filter.
    #    Filter one branch and not the other and excluded documents walk back in.
    embedding_results = self.embedding_search(query, top_k=pool, metadata_filter=metadata_filter)

    return self._reciprocal_rank_fusion(
        [keyword_results, embedding_results],
        top_k=top_k,            # ✏️ cut the wide fused pool back down to what the caller asked for
        k=self.rrf_k,
    )

In [56]:
RetrievalCore.hybrid_search = hybrid_search
print("Patched RetrievalCore.hybrid_search")

Patched RetrievalCore.hybrid_search


In [57]:
suite = TestSuite("Part 2.4 — hybrid_search")

def _rc(n=40, top_k=3):
    """A retriever over n throwaway chunks, each with its own embedding."""
    _chunks = [
        Chunk(document_id="d", index=i, text=f"chunk number {i} about leave policy",
              metadata={"source": "a.md" if i % 2 == 0 else "b.md"})
        for i in range(n)
    ]
    for i, c in enumerate(_chunks):
        c.embedding = [1.0, float(i) / n, 0.0]
    return RetrievalCore(MockEmbedder([1.0, 0.0, 0.0]), chunks=_chunks, top_k=top_k)

@suite.case("hybrid_search", "over-retrieval widens the pool the fuser sees")
def _():
    rc = _rc()
    wide = rc._reciprocal_rank_fusion(
        [rc.keyword_search("leave policy", top_k=60),
         rc.embedding_search("leave policy", top_k=60)], top_k=100)
    narrow = rc._reciprocal_rank_fusion(
        [rc.keyword_search("leave policy", top_k=3),
         rc.embedding_search("leave policy", top_k=3)], top_k=100)
    assert len(wide) > len(narrow)

@suite.case("hybrid_search", "still returns exactly top_k results")
def _():
    assert len(_rc(top_k=3).hybrid_search("leave policy")) == 3

@suite.case("hybrid_search", "overretrieve=1 reproduces the old narrow behaviour")
def _():
    rc = _rc()
    got = rc.hybrid_search("leave policy", overretrieve=1)
    expected = rc._reciprocal_rank_fusion(
        [rc.keyword_search("leave policy", top_k=3),
         rc.embedding_search("leave policy", top_k=3)], top_k=3, k=rc.rrf_k)
    assert [c.id for c, _ in got] == [c.id for c, _ in expected]

@suite.case("hybrid_search", "a pool larger than the corpus degrades gracefully")
def _():
    assert len(_rc(n=2, top_k=3).hybrid_search("leave policy")) == 2

@suite.case("hybrid_search", "metadata_filter reaches both branches")
def _():
    got = _rc().hybrid_search("leave policy", metadata_filter={"source": "a.md"})
    assert got and all(c.metadata["source"] == "a.md" for c, _ in got)

suite.run()

╭──────────────────────────╮
│ Part 2.4 — hybrid_search │
╰──────────────────────────╯

hybrid_search  5/5

✓ over-retrieval widens the pool the fuser sees

✓ still returns exactly top_k results

✓ overretrieve=1 reproduces the old narrow behaviour

✓ a pool larger than the corpus degrades gracefully

✓ metadata_filter reaches both branches

              Summary              
                                   
  Function        Passed   Status  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  hybrid_search    5/5     ✓ PASS

╭─────────────────────────────────────╮
│ All 5 checks passed — nice work! 🎉 │
╰─────────────────────────────────────╯

True

Now see it on the real corpus. Same query, same fusion formula, same `top_k` — the only thing that
changes is how many candidates each branch handed over.

In [58]:
#@title 🔀 Over-retrieval — same fusion, two candidate pools { display-mode: "form" }
_rc_all = RetrievalCore(HashingEmbedder(), chunks=all_chunks, top_k=3)
_q = "how many days of paternity leave"

_panels = []
for _factor in (1, 20):
    _pool = max(3, 3 * _factor)
    _seen = {c.id for c, _ in _rc_all.keyword_search(_q, top_k=_pool)}
    _seen |= {c.id for c, _ in _rc_all.embedding_search(_q, top_k=_pool)}
    _panels.append((f"<code>overretrieve={_factor}</code> · fusion chose from <b>{len(_seen)}</b> candidates",
                    _rc_all.hybrid_search(_q, overretrieve=_factor),
                    "leave_and_absence_policy.md"))

show_compare(_panels, note=f'Hybrid search for "{_q}". Green = the document that answers it.')

The candidate count in each heading is the **ceiling on what fusion could possibly return**. On the
left it is a handful of chunks, so an unrelated policy can hold a top-3 slot simply because nothing
better was in the pool. On the right fusion picks from the whole corpus, and the chunks that both
retrievers liked rise to the top — note the fused scores roughly double, which is what agreement
across two lists looks like.

You'll put a number on that improvement in §2.5.

## 2.5 — Measure it, or you are guessing

You have built three retrievers and made a design decision about each one. So far every claim about
them — "hybrid is more robust", "over-retrieval helps", "chunking beats no chunking" — has been an
*argument*. Arguments are how you generate candidate answers; measurement is how you find out which
ones are true.

The tool is an **evaluation set**: questions whose correct answer you already know, scored the same
way every time. `evaluation.py` ships one — 12 questions over the nine policies in `data/`, each
labelled with the document that answers it, split deliberately into two kinds:

- **lexical** (6) — the question contains the rare exact term the document uses (`MFA`, `PIP`, `MNPI`)
- **semantic** (6) — the question is phrased the way a person actually asks, sharing little vocabulary
  with the document

A single averaged score would hide precisely the difference that matters, so we report both.

In [59]:
for _q in ev.EVAL_SET[:3] + ev.SEMANTIC_QUERIES[:2]:
    print(f"[{_q.kind:8}] {_q.question}")
    print(f"{'':11} → {_q.gold_source}  ({_q.note})")
print(f"\n{len(ev.EVAL_SET)} questions total.")

[lexical ] Is MFA mandatory for VPN connections?
            → information_security_policy.md  (MFA / VPN are near-unique tokens in this corpus)
[lexical ] What is a PIP and when is one issued?
            → performance_review_policy.md  (PIP appears only in the performance policy)
[lexical ] What are the rules about trading while holding MNPI?
            → code_of_conduct.md  (MNPI is a single-document acronym)
[semantic] How much time off do I get each year?
            → leave_and_absence_policy.md  ('time off' vs the document's 'annual leave entitlement')
[semantic] Someone keeps making comments about my background — what do I do?
            → code_of_conduct.md  (describes harassment without using the word)

12 questions total.


### The metrics

**Recall@k** — for what fraction of questions does the gold document appear anywhere in the top `k`?
`k` matters: a RAG system feeds its top few chunks to the LLM, so recall@3 is close to "will the
model actually see the answer?", while recall@1 is the strict version.

That is the standard metric, and on its own it is not enough. Retrieve one un-chunked 1,200-word
document and you "hit" the gold document every time — while handing the LLM ten times the context.
So we measure two more things:

**Answer recall@k** — does the retrieved *text* actually contain the passage that answers the
question? (Checked as a verbatim substring, so the verdict is identical every run.)

**Context cost** — how many words that answer cost you. Context is not free: tokens, money, latency,
and — because models attend worse to long inputs — often accuracy.

Read all three together. The goal is high answer recall at low context cost.

In [60]:
#@title ⚠️ An honest caveat about these numbers { display-mode: "form" }
_rc_eval = RetrievalCore(HashingEmbedder(), chunks=all_chunks, top_k=3)

explain("An honest caveat about these numbers",
        "The notebook runs offline, so the embedding branch uses <code>HashingEmbedder</code> — a "
        "stand-in that scores <b>word overlap</b>, not meaning. It is enough to make the rankings "
        "real (rather than a pile of ties), but a trained embedding model would score much higher on "
        "the semantic half. Trust the <b>shape</b> of what follows, and treat the semantic column as "
        "a floor. There is a cell at the end of this section to re-run everything with real "
        "embeddings if you have a key.",
        tone="warn", icon="⚠️")

### Which retriever wins?

Same corpus, same questions, same `k` — only the strategy changes.

In [61]:
_STRATEGIES = [
    ("Keyword (BM25)", lambda q, k: _rc_eval.keyword_search(q, top_k=k)),
    ("Embedding (cosine)", lambda q, k: _rc_eval.embedding_search(q, top_k=k)),
    ("Hybrid (RRF)", lambda q, k: _rc_eval.hybrid_search(q, top_k=k)),
]
K = 3

_rows = []
for _name, _fn in _STRATEGIES:
    _board = ev.scoreboard(lambda q, f=_fn: f(q, K), k=K)
    _rows.append({"label": _name,
                  "doc": _board["doc_recall"]["overall"],
                  "answer": _board["answer_recall"]["overall"],
                  "words": _board["context_words"]})

show_scoreboard(_rows, title=f"Retrieval strategies at k={K}",
                subtitle="★ marks the best value in each column. Context cost is best when lowest.")

Configuration,Document recall,Answer recall,Context words
Keyword (BM25),0.75,0.50 ★,359
Embedding (cosine),0.58,0.42,338 ★
Hybrid (RRF),0.83 ★,0.42,352


In [62]:
#@title 📊 Recall split by question kind { display-mode: "form" }
_NOTES = {
    "Keyword (BM25)": "Perfect on exact terms, and the gap below it is the problem.",
    "Embedding (cosine)": "The offline stand-in scores word overlap, not meaning — treat this as a floor.",
    "Hybrid (RRF)": "Keeps the lexical score, lifts the semantic one. That gap closing is the point.",
}
for _name, _fn in _STRATEGIES:
    show_kinds(ev.recall_at_k(lambda q, f=_fn: f(q, K), k=K),
               title=f"{_name} · document recall@{K}", subtitle=_NOTES[_name])

Read the two views together, and note that they do **not** tell one clean story.

**Document recall says hybrid wins.** Keyword search is perfect on the lexical half and much weaker
on the semantic half; embedding search (with the offline stand-in) is weaker on both; hybrid keeps
the lexical score *and* lifts the semantic one. That is the real argument for hybrid search — not
that it is best at anything, but that it has no catastrophic weakness, and in production you do not
get to know which kind of question is coming next.

**Answer recall says keyword search wins**, narrowly. Do not explain that away — write it down as
an anomaly to be explained. Two candidate explanations, and they call for different fixes:

1. *Fusion is the problem.* It promotes chunks that **both** lists liked, and a chunk two retrievers
   agree on is not always the chunk holding the exact sentence. If so, hybrid search has a real
   precision cost you would need to design around.
2. *A branch is the problem.* Fusion has no judgement of its own — it can only reorder what its
   inputs supply. Feed it a weak retriever and it will faithfully mix that retriever's noise into
   the result. If so, nothing is wrong with fusion; the embedding branch is simply bad here.

Notice that the offline embedding row scores 0.33 on the semantic half — it is, on this evidence,
the weak retriever. So (2) is the better bet. The last cell of this section settles it by swapping in
a real embedding model and re-running exactly the same measurement.

This is the loop worth taking away from the whole notebook: **measure → find something that does not
fit → name the competing explanations → design the measurement that separates them.**

Which questions still fail?

In [63]:
#@title 🎯 Which questions still fail { display-mode: "form" }
show_eval_grid(ev.per_query_hits(lambda q: _rc_eval.hybrid_search(q, top_k=K), k=K),
               title=f"Hybrid search · document recall@{K}",
               subtitle="A recall number tells you how many failed. This tells you which — and what came back instead.")

### Does over-retrieval actually help?

§2.4 argued that fusion needs a wide candidate pool. Here is the claim, measured. Only
`overretrieve` changes:

In [64]:
_sweep = []
for _factor in (1, 2, 5, 20, 100):
    _score = ev.recall_at_k(lambda q, f=_factor: _rc_eval.hybrid_search(q, top_k=K, overretrieve=f), k=K)
    _sweep.append((f"{_factor}×", _score["overall"], None))

show_sweep(_sweep, title=f"Hybrid document recall@{K} vs over-retrieval factor",
           x_label="overretrieve (candidates per branch = top_k × this)",
           subtitle="overretrieve=1 is the narrow version: each branch hands over only top_k.")

Real, and it plateaus. Widening the pool buys you accuracy up to a point, after which fusion already
has every candidate that matters and more does nothing. That plateau is *why* a default like 20 is
reasonable, and it is the kind of thing you can only learn by measuring.

### How big should a chunk be?

`chunk_size` and `overlap` are the highest-leverage knobs in the whole system — and until now they
have been defaults nobody questioned. Re-ingest the corpus at several sizes and score each one.
(This runs the full ingestion pipeline once per configuration, so give it a few seconds.)

In [65]:
def evaluate_config(label, **ingest_kwargs):
    """Ingest the corpus with these settings, then score retrieval over it."""
    _emb = HashingEmbedder()
    _core = IngestionCore(_emb, DBManager(), collection_name="sweep", **ingest_kwargs)
    _chunks = []
    for d in docs:
        _chunks.extend(_core.ingest_document(d))
    _rc = RetrievalCore(_emb, chunks=_chunks, top_k=K)
    _board = ev.scoreboard(lambda q: _rc.hybrid_search(q, top_k=K), k=K)
    return {"label": f"{label} · {len(_chunks)} chunks",
            "doc": _board["doc_recall"]["overall"],
            "answer": _board["answer_recall"]["overall"],
            "words": _board["context_words"]}

_configs = [
    evaluate_config("no chunking", strategy="whole_document"),
    evaluate_config("size 40 / overlap 10", chunk_size=40, overlap=10),
    evaluate_config("size 120 / overlap 20", chunk_size=120, overlap=20),
    evaluate_config("size 250 / overlap 40", chunk_size=250, overlap=40),
    evaluate_config("size 600 / overlap 80", chunk_size=600, overlap=80),
]
show_scoreboard(_configs, title=f"Chunking configurations at k={K}",
                subtitle="Same retriever, same questions — only how the corpus was split changes.")

Connected to Qdrant (in-memory mode)
Collection 'sweep' created (vector size: 128)
Inserted 1 chunks into 'sweep'
Inserted 1 chunks into 'sweep'
Inserted 1 chunks into 'sweep'
Inserted 1 chunks into 'sweep'
Inserted 1 chunks into 'sweep'
Inserted 1 chunks into 'sweep'
Inserted 1 chunks into 'sweep'
Inserted 1 chunks into 'sweep'
Inserted 1 chunks into 'sweep'
Connected to Qdrant (in-memory mode)
Collection 'sweep' created (vector size: 128)
Inserted 34 chunks into 'sweep'
Inserted 42 chunks into 'sweep'
Inserted 6 chunks into 'sweep'
Inserted 32 chunks into 'sweep'
Inserted 39 chunks into 'sweep'
Inserted 37 chunks into 'sweep'
Inserted 41 chunks into 'sweep'
Inserted 36 chunks into 'sweep'
Inserted 6 chunks into 'sweep'
Connected to Qdrant (in-memory mode)
Collection 'sweep' created (vector size: 128)
Inserted 10 chunks into 'sweep'
Inserted 13 chunks into 'sweep'
Inserted 2 chunks into 'sweep'
Inserted 10 chunks into 'sweep'
Inserted 12 chunks into 'sweep'
Inserted 11 chunks into 'sw

Configuration,Document recall,Answer recall,Context words
no chunking · 9 chunks,0.83 ★,0.83 ★,"3,218"
size 40 / overlap 10 · 273 chunks,0.75,0.33,119 ★
size 120 / overlap 20 · 84 chunks,0.83 ★,0.42,352
size 250 / overlap 40 · 41 chunks,0.75,0.50,694
size 600 / overlap 80 · 19 chunks,0.75,0.58,"1,441"


Read that table carefully, because the obvious conclusion is wrong.

**"No chunking" wins on both recall columns** — and costs roughly ten times the context. That is not
a tie: you are paying for a whole document to deliver one paragraph, on every single query. Document
recall alone literally *cannot* see this, which is exactly why we measure answer recall and context
cost as well.

At the other end, very small chunks are cheap but score badly — an answer that spans a boundary is
in no single chunk, so it never arrives whole. Which is what `overlap` is for:

In [66]:
_overlaps = [evaluate_config(f"overlap {_ov}", chunk_size=120, overlap=_ov) for _ov in (0, 20, 40, 60)]
show_scoreboard(_overlaps, title=f"Overlap at chunk_size=120, k={K}",
                subtitle="Overlap costs a few percent more chunks. What does it buy?")

Connected to Qdrant (in-memory mode)
Collection 'sweep' created (vector size: 128)
Inserted 9 chunks into 'sweep'
Inserted 11 chunks into 'sweep'
Inserted 2 chunks into 'sweep'
Inserted 8 chunks into 'sweep'
Inserted 10 chunks into 'sweep'
Inserted 10 chunks into 'sweep'
Inserted 11 chunks into 'sweep'
Inserted 10 chunks into 'sweep'
Inserted 2 chunks into 'sweep'
Connected to Qdrant (in-memory mode)
Collection 'sweep' created (vector size: 128)
Inserted 10 chunks into 'sweep'
Inserted 13 chunks into 'sweep'
Inserted 2 chunks into 'sweep'
Inserted 10 chunks into 'sweep'
Inserted 12 chunks into 'sweep'
Inserted 11 chunks into 'sweep'
Inserted 13 chunks into 'sweep'
Inserted 11 chunks into 'sweep'
Inserted 2 chunks into 'sweep'
Connected to Qdrant (in-memory mode)
Collection 'sweep' created (vector size: 128)
Inserted 13 chunks into 'sweep'
Inserted 16 chunks into 'sweep'
Inserted 2 chunks into 'sweep'
Inserted 12 chunks into 'sweep'
Inserted 15 chunks into 'sweep'
Inserted 14 chunks int

Configuration,Document recall,Answer recall,Context words
overlap 0 · 73 chunks,0.75,0.42,331 ★
overlap 20 · 84 chunks,0.83 ★,0.42,352
overlap 40 · 103 chunks,0.75,0.58 ★,343
overlap 60 · 130 chunks,0.67,0.58 ★,359


Overlap buys **answer recall at almost no extra context cost** — the answer that used to fall
between two chunks now sits whole inside one of them. That is the entire justification for the
parameter, and now it is a measurement rather than a story.

> **Your turn to experiment.** Change `K` above and re-run. Try `top_k=10` — recall goes up, and so
> does context cost. There is no globally correct setting; there is a tradeoff curve, and now you
> have the instrument to find where you want to sit on it.

In [67]:
#@title 🔬 What this evaluation set is not { display-mode: "form" }
explain("What this evaluation set is not",
        "Twelve hand-written questions are enough to see a trend and catch a regression. They are "
        "<b>not</b> enough to certify a system. Real evaluation sets are larger, are built from "
        "questions users actually asked, and are refreshed as the corpus changes — otherwise you end "
        "up optimising for twelve questions instead of for your users. The habit is what matters: "
        "change one thing, measure, keep it or revert it.",
        tone="warn", icon="🔬")

### Settling it: re-run with a real embedding model

Everything above ran on `HashingEmbedder`, which scores word overlap rather than meaning. If
explanation (2) is right — fusion is fine, the embedding *branch* was weak — then swapping in a
trained embedding model should lift the embedding row **and** carry hybrid's answer recall up with
it, with no change to the fusion code at all.

That is a real prediction, and it is falsifiable. Run the cell (needs the key from §0.2) and see.

In [68]:
if HAS_KEY:
    from clients.embedder import TextEmbedder

    _real = TextEmbedder(api_key=os.environ["OPENROUTER_API_KEY"])
    _core_real = IngestionCore(_real, DBManager(), collection_name="real",
                               chunk_size=120, overlap=20)
    _real_chunks = []
    for d in docs:
        _real_chunks.extend(_core_real.ingest_document(d))
    _rc_real = RetrievalCore(_real, chunks=_real_chunks, top_k=K)

    _real_rows = []
    for _name, _fn in [("Keyword (BM25)", _rc_real.keyword_search),
                       ("Embedding (real)", _rc_real.embedding_search),
                       ("Hybrid (RRF)", _rc_real.hybrid_search)]:
        _b = ev.scoreboard(lambda q, f=_fn: f(q, top_k=K), k=K)
        _real_rows.append({"label": _name, "doc": _b["doc_recall"]["overall"],
                           "answer": _b["answer_recall"]["overall"], "words": _b["context_words"]})

    show_scoreboard(_rows, title=f"Offline stand-in · k={K}",
                    subtitle="HashingEmbedder — word overlap, no meaning. (Repeated here to compare.)")
    show_scoreboard(_real_rows, title=f"Real embedding model · k={K}",
                    subtitle="Identical corpus, identical questions, identical fusion code.")
    show_kinds(ev.recall_at_k(lambda q: _rc_real.embedding_search(q, top_k=K), k=K),
               title="Real embedding search · document recall",
               subtitle="Compare against the 'Embedding (cosine)' bars earlier in this section.")
else:
    print("No OPENROUTER_API_KEY set — skipping.")
    print("The offline results above still stand; the paragraph below tells you what this cell shows.")

Connected to Qdrant (in-memory mode)
Collection 'real' created (vector size: 3072)
Inserted 10 chunks into 'real'
Inserted 13 chunks into 'real'
Inserted 2 chunks into 'real'
Inserted 10 chunks into 'real'
Inserted 12 chunks into 'real'
Inserted 11 chunks into 'real'
Inserted 13 chunks into 'real'
Inserted 11 chunks into 'real'
Inserted 2 chunks into 'real'


Configuration,Document recall,Answer recall,Context words
Keyword (BM25),0.75,0.50 ★,359
Embedding (cosine),0.58,0.42,338 ★
Hybrid (RRF),0.83 ★,0.42,352


Configuration,Document recall,Answer recall,Context words
Keyword (BM25),0.75,0.50,359
Embedding (real),1.00 ★,0.83,338 ★
Hybrid (RRF),1.00 ★,0.92 ★,356


**Prediction confirmed, and then some.** With a real embedding model the embedding branch goes from
0.33 to **1.00** on the semantic half — the questions BM25 could never reach are exactly the ones a
trained embedder handles — and hybrid's answer recall rises from 0.42 to **0.92**, the best of the
three. Explanation (2) was right: fusion was never the problem, it was faithfully mixing in a weak
branch's noise.

Three things worth taking from that:

- **Fusion inherits the quality of its inputs.** RRF has no judgement of its own. "Add hybrid search"
  is not a fix for a bad retriever; it is a way to combine good ones.
- **Your conclusion is only as good as your measurement setup.** Every number in this section was
  correct, and the story they told was still misleading, because one component was a stand-in. When
  you report a result, report what was mocked.
- **The eval set is now saturated** — hybrid gets 12/12, so it can no longer tell a good
  configuration from a great one. That is not a victory, it is a signal: a set everything passes has
  stopped measuring. The next move is harder questions, not a better score.

## 2.6 — Putting it together: answer with the LLM

The final step of RAG: **retrieve**, stuff the chosen chunk texts into a **context block**, and ask
the LLM to answer *using only that context*. You return both the answer and the source chunks (the
web app shows them as citations).

Provided on `self`: `self._retrieve(query, search_type, metadata_filter)` (dispatches to the right
search, and applies the `min_score` threshold), `self.llm.complete(prompt, system_prompt=...)`, and
`self.system_prompt`.

**One thing before the happy path: know when *not* to answer.** Retrieval always returns something —
the nearest chunks exist even when nothing in the corpus is relevant. If you hand those to the LLM
anyway, you get a confident answer built from the least-bad text in your store, which is the single
most damaging failure mode a RAG system has. So: no results, no answer. Return the module-level
`NO_CONTEXT_ANSWER` instead, and skip the LLM call entirely.

In [69]:
#@title ✏️ Your task · 2.6 { display-mode: "form" }
from IPython.display import HTML, display
display(HTML('<div style="border:1px solid #e3e8ef;border-left:5px solid #0f766e;border-radius:12px;\n     background:linear-gradient(135deg,#fbfefe,#f7fafc);padding:0 0 16px;margin:14px 0;\n     font-family:ui-sans-serif,system-ui,sans-serif;max-width:920px">\n  <div style="background:#0f766e;color:#fff;padding:7px 16px;border-radius:7px 7px 0 0;\n       font-size:11px;font-weight:700;letter-spacing:.09em;text-transform:uppercase">\n    ✏️ Your task &nbsp;·&nbsp; 2.6\n  </div>\n  <div style="padding:0 16px">\n    <div style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:13.5px;color:#1f2933;font-weight:600;\n         background:#fff;border:1px solid #e3e8ef;border-radius:7px;padding:8px 11px;margin-top:13px;\n         overflow-x:auto;white-space:pre">retrieve_and_answer(self, query, search_type=None, metadata_filter=None)</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">What it must do</div>\n    <div style="font-size:13px;color:#1f2933;line-height:1.55">The last step of RAG: retrieve, paste the chosen chunk texts into a context block, and ask the LLM to answer using only that context. Return the answer <b>and</b> the chunks it came from, so the app can cite them.</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Use these — already provided</div>\n    <table style="border-collapse:collapse;width:100%"><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">self._retrieve(query, search_type, metadata_filter)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">ranked results, min_score already applied</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">NO_CONTEXT_ANSWER</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the abstention answer, when nothing was retrieved</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">"\\n\\n".join(chunk.text for chunk, _ in results)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the context block</td></tr><tr><td style="padding:3px 10px 3px 0;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:11.5px;color:#0f766e;white-space:nowrap;vertical-align:top;text-align:left">self.llm.complete(prompt, system_prompt=self.system_prompt)</td><td style="padding:3px 0;font-size:12px;color:#334155;vertical-align:top;text-align:left">the generated answer</td></tr></table>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Returns</div>\n    <div style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px;color:#1f2933">tuple[str, list[tuple[Chunk, float]]] — (answer, sources)</div>\n    <div style="font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#647084;margin:12px 0 4px">Where it plugs in</div>\n    <div style="font-size:12px;color:#334155">Once written you bind it onto\n      <code style="font-family:ui-monospace,SFMono-Regular,Menlo,monospace;color:#0f766e">RAGCore.retrieve_and_answer</code>, and the whole pipeline\n      starts using your version. The cell after it checks your work.</div>\n    <div style="margin-top:10px;padding:7px 10px;background:#fffbeb;border:1px solid #fde68a;border-radius:7px;font-size:12px;color:#78350f">⚠️ Check for empty results <b>before</b> calling the LLM. Answering from an empty context is how a RAG system produces a confident, fluent, completely unsupported answer.</div>\n  </div>\n</div>'))

"self._retrieve(query, search_type, metadata_filter)","ranked results, min_score already applied"
NO_CONTEXT_ANSWER,"the abstention answer, when nothing was retrieved"
"""\n\n"".join(chunk.text for chunk, _ in results)",the context block
"self.llm.complete(prompt, system_prompt=self.system_prompt)",the generated answer


In [70]:
def retrieve_and_answer(self, query, search_type=None, metadata_filter=None):
    """Retrieve, then answer with the LLM. Return (answer_str, list[(Chunk, score)])."""
    results = self._retrieve(query, search_type, metadata_filter)
    if not results:
        return NO_CONTEXT_ANSWER, []                                        # ✏️ abstain — and note the LLM call below is never reached

    context = "\n\n".join(chunk.text for chunk, _score in results)          # ✏️ the retrieved passages, as one context block
    prompt = f"Context:\n{context}\n\nQuestion: {query}"
    answer = self.llm.complete(prompt, system_prompt=self.system_prompt)    # ✏️ the pipeline's system prompt
    return answer, results

In [71]:
RAGCore.retrieve_and_answer = retrieve_and_answer
print("Patched RAGCore.retrieve_and_answer")

Patched RAGCore.retrieve_and_answer


In [72]:
suite = TestSuite("Part 2.6 — retrieve_and_answer")

# A full RAGCore wired to mocks: no network, fully deterministic.
_rag = RAGCore(MockEmbedder([1.0, 0.0, 0.0]), MockLLM(), chunk_size=5, overlap=1)
_rag.ingest_document(Document(
    text="employees are entitled to twenty five days of annual leave per year",
    metadata={"source": "leave.md", "document_title": "Leave"},
))

@suite.case("retrieve_and_answer", "returns an answer plus the source chunks")
def _():
    answer, sources = _rag.retrieve_and_answer("annual leave days", search_type=SearchType.KEYWORD)
    assert isinstance(answer, str) and answer
    assert len(sources) >= 1

@suite.case("retrieve_and_answer", "the retrieved context reaches the LLM prompt")
def _():
    _rag.retrieve_and_answer("annual leave days", search_type=SearchType.KEYWORD)
    assert "annual leave" in _rag.llm.last_prompt   # MockLLM records what it was sent

@suite.case("retrieve_and_answer", "abstains when retrieval comes back empty")
def _():
    empty = RAGCore(MockEmbedder([1.0, 0.0, 0.0]), MockLLM())   # nothing ingested
    answer, sources = empty.retrieve_and_answer("anything at all")
    assert answer == NO_CONTEXT_ANSWER and sources == []

@suite.case("retrieve_and_answer", "abstaining does not call the LLM")
def _():
    empty = RAGCore(MockEmbedder([1.0, 0.0, 0.0]), MockLLM())
    empty.retrieve_and_answer("anything at all")
    assert empty.llm.last_prompt is None, "there was no context to ask about"

suite.run()

Connected to Qdrant (in-memory mode)
Collection 'documents' created (vector size: 3)
Inserted 3 chunks into 'documents'


╭────────────────────────────────╮
│ Part 2.6 — retrieve_and_answer │
╰────────────────────────────────╯

Connected to Qdrant (in-memory mode)
Connected to Qdrant (in-memory mode)


retrieve_and_answer  4/4

✓ returns an answer plus the source chunks

...

✓ abstains when retrieval comes back empty

✓ abstaining does not call the LLM

                 Summary                 
                                         
  Function              Passed   Status  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  retrieve_and_answer    4/4     ✓ PASS

╭─────────────────────────────────────╮
│ All 4 checks passed — nice work! 🎉 │
╰─────────────────────────────────────╯

True

**Abstention in the wild.** The interesting case is not an empty store — it is a *full* one, asked
something it has no answer to. Retrieval still returns the three nearest chunks, because the nearest
chunks always exist:

In [73]:
#@title 🛑 One question it can answer, one it cannot { display-mode: "form" }
_qa = RAGCore(HashingEmbedder(), MockLLM(), search_type=SearchType.KEYWORD,
              chunk_size=120, overlap=20)
_qa.ingest_path("data")

_off = "sourdough levain hydration autolyse"
_on = "Is MFA mandatory for VPN connections?"

show_compare([
    (f'A question the corpus <b>can</b> answer:<br><code>"{_on}"</code>',
     _qa.retrieval_core.keyword_search(_on, top_k=3), "information_security_policy.md"),
    (f'A question it <b>cannot</b>:<br><code>"{_off}"</code>',
     _qa.retrieval_core.keyword_search(_off, top_k=3), None),
], note="Look at the scores, not the text. That is the signal a threshold uses.")

Connected to Qdrant (in-memory mode)
Collection 'documents' created (vector size: 128)
Inserted 10 chunks into 'documents'
Inserted 13 chunks into 'documents'
Inserted 2 chunks into 'documents'
Inserted 10 chunks into 'documents'
Inserted 12 chunks into 'documents'
Inserted 11 chunks into 'documents'
Inserted 13 chunks into 'documents'
Inserted 11 chunks into 'documents'
Inserted 2 chunks into 'documents'


Two very different situations, and by default they produce the same shape of answer: three chunks
and a confident reply. `min_score` is the threshold that separates them.

In [74]:
_qa.min_score = 3.0          # BM25 scale: real questions here score 7–28, nonsense scores 0
_answer, _sources = _qa.retrieve_and_answer(_off)
show_answer(_answer, _sources)

explain("Why this matters more than it looks",
        "Without a threshold the LLM receives three chunks about leave policy and a question about "
        "bread, and a helpful model will find <i>something</i> to say. The answer comes out fluent, "
        "sourced, and wrong — the worst combination a RAG system can produce. A threshold plus an "
        "explicit abstention is what makes \"I don't know\" a first-class answer.",
        tone="good", icon="🛑")

Now the part most tutorials skip: **choosing that number is the hard bit, and it is fragile.** Watch
the same nonsense question survive the same threshold, purely because it was phrased as a sentence:

In [75]:
#@title ⚠️ When the threshold does not save you { display-mode: "form" }
_off_sentence = "What is the airspeed velocity of an unladen swallow?"
_top = _qa.retrieval_core.keyword_search(_off_sentence, top_k=1)
print(f'"{_off_sentence}"\n  BM25 top score: {_top[0][1]:.2f}  (threshold is {_qa.min_score})')
print(f'"{_off}"\n  BM25 top score: {_qa.retrieval_core.keyword_search(_off, top_k=1)[0][1]:.2f}')

explain("The stopwords did it",
        "Both questions are equally unanswerable, but the sentence shares <code>what · is · the · "
        "of · an</code> with every document in the corpus. We do no stopword removal, so those "
        "common tokens contribute real (if small) score, and across a long chunk they add up to more "
        "than the threshold.<br><br>"
        "Three honest conclusions: <b>(1)</b> the right threshold is corpus- and strategy-specific — "
        "cosine lives in [-1, 1], BM25 is unbounded, RRF clusters near 1/60, so a number that works "
        "for one is meaningless for another. <b>(2)</b> Set it by measuring, exactly as in §2.5: too "
        "high and you abstain on questions you could have answered. <b>(3)</b> A trained embedding "
        "model separates on-topic from off-topic far more cleanly than raw BM25 does — this is one of "
        "the places where the offline stand-in is genuinely worse than the real thing.",
        tone="warn", icon="⚠️")

"What is the airspeed velocity of an unladen swallow?"
  BM25 top score: 7.55  (threshold is 3.0)
"sourdough levain hydration autolyse"
  BM25 top score: 0.00


**See it rendered.** The mock LLM just echoes the prompt it received, so this view doubles as an
X-ray of *exactly* what your pipeline sends the model — the retrieved context plus the question.

In [76]:
#@title 💬 The full prompt your pipeline sends the model { display-mode: "form" }
_ans, _src = _rag.retrieve_and_answer("how many days of annual leave", search_type=SearchType.KEYWORD)
show_answer(_ans, _src)

### Live demo *(optional — needs an OpenRouter key)*

If you set a key in 0.2, this ingests the real policy folder and answers a question with the actual
embedding + LLM models. Otherwise it is skipped.

In [77]:
if HAS_KEY:
    from clients.embedder import TextEmbedder
    from clients.llm import LLMClient
    live = RAGCore(TextEmbedder(api_key=os.environ["OPENROUTER_API_KEY"]),
                   LLMClient(api_key=os.environ["OPENROUTER_API_KEY"]))
    live.ingest_path("data")
    answer, sources = live.retrieve_and_answer(
        "Can I accept a CHF 120 gift from a vendor?", search_type=SearchType.HYBRID)
    show_answer(answer, sources)
else:
    print("No API key set — skipping the live demo (the offline tests above already prove it works).")

Connected to Qdrant (in-memory mode)
Collection 'documents' created (vector size: 3072)
Inserted 7 chunks into 'documents'
Inserted 8 chunks into 'documents'
Inserted 1 chunks into 'documents'
Inserted 6 chunks into 'documents'
Inserted 8 chunks into 'documents'
Inserted 7 chunks into 'documents'
Inserted 8 chunks into 'documents'
Inserted 7 chunks into 'documents'
Inserted 1 chunks into 'documents'


## 2.7 — Export your Part 2 functions

Writes your retrieval functions to `solutions/part2_retrieval.py`. **Run your implementation cells
above first.**

In [78]:
HEADER_P2 = (
    '"""Part 2 — exported from the notebook. Do not edit by hand; re-export instead."""\n'
    'from __future__ import annotations\n'
    'import numpy as np\n'
    'from rank_bm25 import BM25Okapi\n'
    'from rag_core import NO_CONTEXT_ANSWER\n'
    'from retrieval_core import _tokenize\n\n\n'
)

_funcs_p2 = [cosine_similarity, keyword_search, reciprocal_rank_fusion,
             hybrid_search, retrieve_and_answer]

# Same guard as Part 1: only fully-filled functions are exported. Anything skipped here
# stays unimplemented in the app until you come back and finish it.
_ready_p2 = [f for f in _funcs_p2 if not has_blanks(f)]
for f in _funcs_p2:
    if has_blanks(f):
        print(f"⚠️  Skipping {f.__name__} — it still has unfilled TODO blanks.")

_body = HEADER_P2 + "\n\n".join(inspect.getsource(f) for f in _ready_p2)

_path = pathlib.Path("solutions/part2_retrieval.py")
_path.parent.mkdir(exist_ok=True)
_path.write_text(_body, encoding="utf-8")
print(f"Wrote {_path} ({len(_ready_p2)}/{len(_funcs_p2)} functions).")

try:
    from google.colab import files
    files.download(str(_path))
except Exception:
    pass

Wrote solutions\part2_retrieval.py (5/5 functions).


---
# Part 3 — Run the full application *(optional)*

You've built the engine. The repo also ships a small **web app** around it so you can try your RAG
system in a browser.

## How it fits together

- **Backend** (`server.py`, FastAPI): wraps your pipeline in an HTTP API. It holds one shared
  persistent vector store and builds a fresh `RAGCore` per request. Each request carries *your own*
  OpenRouter key in a header — the server never stores it.
- **Frontend** (`frontend/`, React + Vite): a key field, document upload, a question box with a
  strategy selector, and an answer view with expandable source citations.
- **Your code plugs in via `solutions.apply()`**: on startup the app monkey-patches the two files
  you exported (`solutions/part1_ingestion.py`, `solutions/part2_retrieval.py`) onto the real
  modules — exactly what you did cell-by-cell here. The repo ships **no** implementation of those
  eight functions, so this is not a nicety: until both files are in place the app boots but cannot
  answer a question. Its **Implementation** panel shows exactly which ones are still missing.

## Steps

1. Clone the course repo and go to the project folder — everything below is relative to it:
   ```bash
   git clone https://github.com/eth-fdd-fs26/FDD-WE5-public.git
   cd FDD-WE5-public/project
   ```
2. Make sure your exported files are in that folder's `solutions/` (Parts 1.4 and 2.7). In Colab
   the export cells also trigger a browser download, so you can drop them straight in.
3. **Backend** (from `project/`):
   ```bash
   uv sync                       # or: pip install -e .
   uv run uvicorn server:app --reload
   ```
4. **Frontend** (another terminal, also from `project/`):
   ```bash
   cd frontend && npm install && npm run dev
   ```
5. Open the printed URL, paste your OpenRouter key, upload the policies in `data/`, and ask away.

> The backend prints `[solutions] applied your implementations: ...` at startup when it picks up
> your exported files — a quick way to confirm your code is the one running.

---
## Appendix — write your own tests

Use this scratch space to probe anything you're unsure about — print intermediate values, try edge
cases, or add `TestSuite` checks of your own. The same harness the graders use:

In [79]:
suite = TestSuite("My experiments")

@suite.case("sliding_window", "my own edge case")
def _():
    # ✏️ change me
    assert sliding_window("a b c d", chunk_size=2, overlap=0) == ["a b", "c d"]

suite.run()

# ...or just scratch:
# print(chunking.chunk_text(docs[0].text, chunk_size=50, overlap=10)[:2])

╭────────────────╮
│ My experiments │
╰────────────────╯

sliding_window  1/1

✓ my own edge case

              Summary               
                                    
  Function         Passed   Status  
 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 
  sliding_window    1/1     ✓ PASS

╭─────────────────────────────────────╮
│ All 1 checks passed — nice work! 🎉 │
╰─────────────────────────────────────╯

True